# 📘 Working Note: Target-Free TVT Geosteering in ROGII

This Working Note was prepared by **Pilkwang Kim** on **July 4, 2026 at 05:10 UTC**.

## Executive summary

This note reframes the ROGII TVT task as **target-free geosteering** rather than ordinary tabular regression.

Main conclusions:

1. The last-known `TVT_input` anchor is strong, but incomplete.
2. Prefix slope extrapolation is fragile and should not be trusted globally.
3. GR/typewell matching is a likelihood signal, not a hard depth label.
4. Heel-calibrated GR/contact diagnostics show that the datum is mechanically recoverable for many train-side wells, while same-well contact use remains a guarded public-overlap policy rather than private-safe evidence.
5. After datum and tie handling, hidden-tail shape, slope, and curvature remain real but secondary.
6. Validation must evaluate the composed prediction, including selectors, projection, guards, and postprocess.

| Question | Short answer |
|---|---|
| What is predicted? | `TVT` along the hidden tail of each horizontal well. |
| What is visible? | Full trajectory, GR log, typewell curves, and known-prefix `TVT_input`. |
| What is dangerous? | Hidden-tail target summaries, row-random validation, and hard reforks based only on public score noise. |
| What remains open? | Per-well shape/slope after datum recovery and two-mode ambiguity are handled. |

A useful mental model is:

$$
\text{TVT error} \approx
\underbrace{\text{datum / offset error}}_{\text{mostly recoverable, MSE-dominant when missed}}
+
\underbrace{\text{two-mode bundle ambiguity}}_{\text{detect, then hedge}}
+
\underbrace{\text{per-well shape / slope error}}_{\text{open frontier}}.
$$


In [ ]:
from pathlib import Path
from IPython.display import Image, display

PUBLIC_FIGURE_ROOT = Path('/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures')

def _figure_roots():
    roots = [PUBLIC_FIGURE_ROOT]
    for base in [Path.cwd(), *Path.cwd().parents]:
        roots.extend([base / 'Fig_WorkingNotes', base / 'Fig_Graph'])
    seen = set()
    unique = []
    for root in roots:
        key = str(root)
        if key not in seen:
            seen.add(key)
            unique.append(root)
    return unique

FIG_ROOTS = _figure_roots()

def show_public_figure(name, width=None):
    for root in FIG_ROOTS:
        path = root / name
        if path.exists():
            display(Image(filename=str(path), width=width))
            return path
    print(f'Figure file not found: {name}')
    return None


**Figure E. Three operating profiles for the ROGII system.**  
The same evidence base supports three different operating modes: public-aggressive reproduction, hybrid robustness, and private-safe unseen-well generalization. The working note separates these profiles instead of treating every strong public signal as private-safe evidence.

In [ ]:
show_public_figure('ROGII_WorkingNote_FIG_E.png')

<!-- FIG_E_READING_GUIDE -->
**How to read Figure E.**  The three profiles are not three different interpretations of the data; they are three different policy choices about which evidence is allowed to move the prediction.  A compact way to write the tradeoff is

$$
\hat T_i^{profile}
= \hat T_i^{path}
+ g_i^{profile}\,\Delta_i^{model}
+ h_i^{profile}\,\Delta_i^{overlap},
$$

where $\hat T_i^{path}$ is the target-free path estimate, $\Delta_i^{model}$ is a learned correction, and $\Delta_i^{overlap}$ is a guarded public-overlap correction.  The private-safe profile keeps $h_i^{profile}=0$ and makes $g_i^{profile}$ small and evidence-gated.  The public-aggressive profile may allow $h_i^{profile}>0$, but only under explicit guardrails.

In [ ]:
show_public_figure('ROGII_Graph_FigMain.png')

**Figure. Target-free TVT geosteering frame.**  The note treats the hidden interval as a stratigraphic path recovery problem: use the prefix anchor, observed trajectory, GR/typewell likelihood, and structural priors without reading hidden-tail TVT.

In [ ]:

import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator, FuncFormatter
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

def _find_local_repo_root() -> Path:
    env_root = os.environ.get('ROGII_LOCAL_ROOT')
    if env_root:
        return Path(env_root).expanduser().resolve()
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'sample_submission.csv').exists() and (candidate / 'train').exists():
            return candidate.resolve()
    return Path.cwd().resolve()

LOCAL_REPO_ROOT = _find_local_repo_root()
KAGGLE_DATA_ROOTS = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
]
DATA_ROOT = next(
    (root for root in KAGGLE_DATA_ROOTS if (root / 'train').exists()),
    LOCAL_REPO_ROOT,
)
TRAIN_DIR = DATA_ROOT / 'train'
TEST_DIR = DATA_ROOT / 'test'
OUT_DIR = Path('/kaggle/working/error_anatomy') if Path('/kaggle/working').exists() else LOCAL_REPO_ROOT / 'outputs' / 'error_anatomy'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT:', DATA_ROOT)
print('OUT_DIR:', OUT_DIR)


## Notebook roadmap

> **Skim path.** Read Sections 0-3 for the problem setup, Sections 4-10 for the EDA evidence, and Sections 11-16 for the error model and next-step logic.

| Layer | Sections | What to look for |
|---|---:|---|
| Problem frame | 0-1 | Vocabulary, leakage boundary, and why the unit is a well. |
| Visible-object EDA | 2-10 | Prefix anchors, hidden-tail length, GR/typewell signal, spatial structure, and baseline behavior. |
| Error anatomy | 11-16 | Datum vs mode ambiguity vs residual shape/slope. |
| Credit trail | 17 | Public notebooks and discussions that shaped the note. |

The split is intentional.  Submission tactics changed quickly during the competition; these diagnostics are meant to remain useful after leaderboard-specific reforks move on.

The appendix contains compact policy/action ledgers.  They separate measured facts, diagnostic claims, public-overlap hypotheses, and the modeling actions that follow from each one without interrupting the core EDA narrative.


## 0. How the discussion evolved

The competition discussion moved from row prediction toward well-level geosteering.  The useful history is easier to read as a sequence of pivots.

| Stage | What changed | What survived |
|---|---|---|
| Row-regression intuition | Early GBDT baselines treated rows as tabular samples. | Useful sanity check, but weak at well-level datum and slope errors. |
| Target-free alignment | The typewell became a vertical GR reference and the horizontal trace became an observed path through space. | Candidate TVT paths can be generated from geometry, GR alignment, prefix calibration, PF/beam search, or surface priors. |
| Leakage boundary | The question became whether a signal exists at inference time. | Prefix `TVT_input`, geometry, GR, and typewell curves are legal; hidden-tail targets and row-random validation are not. |
| Reforks and mode ambiguity | Public notebooks found strong physical/ridge/PF recipes, but small score changes often lived inside a narrow stochastic band. | The best-looking GR fit can still land on the wrong datum. |
| Current working hypothesis | The remaining task is not one monolithic model error. | Recover datum when evidence is strong, hedge when modes are ambiguous, and measure residual shape/slope directly. |


**Figure B. From tabular regression to target-free geosteering.**  
This note treats TVT prediction as hidden-tail stratigraphic path recovery. Instead of assigning TVT independently row by row, the approach uses the prefix anchor, trajectory geometry, GR/typewell likelihood, and formation-surface priors to infer a coherent TVT path.

In [ ]:
show_public_figure('ROGII_WorkingNote_FIG_B.png')

<!-- FIG_B_READING_GUIDE -->
**How to read Figure B.**  A row model answers a local question,

$$
\hat T_{w,i}=f(x_{w,i}),
$$

as if each row were exchangeable.  Target-free geosteering instead estimates one coherent hidden-tail function for the well,

$$
\hat T_w(s)=\hat D_w+\hat\phi_w(s),\qquad s\in[0,1],
$$

where $\hat D_w$ is the datum implied by prefix anchors and geology, and $\hat\phi_w(s)$ is the tail shape constrained by trajectory, GR/typewell likelihood, and formation structure.  This is why the notebook repeatedly evaluates wells and paths rather than isolated rows.

## 1. Vocabulary and information boundary

| Term | Meaning in this competition |
|---|---|
| `MD` | Measured depth along the drilled wellbore path. |
| `X/Y/Z` | Spatial coordinates of the row; `Z` carries vertical position. |
| `GR` | Gamma-ray log, a lithology-sensitive stratigraphic barcode. |
| Typewell | Vertical reference well where `TVT -> GR` is known. |
| Horizontal well | Target well.  The prefix has known `TVT_input`; the tail must be predicted. |
| `TVT` | True vertical thickness coordinate, the hidden target for tail rows. |
| `TVT_input` | Legal known-prefix target coordinate, missing in the prediction zone. |

| Evidence source | Use | Caution |
|---|---|---|
| `MD`, `X`, `Y`, `Z`, `GR` | Observable covariates in train and test. | Future covariates are batch-available, but target-free only. |
| Prefix `TVT_input` | Anchor, prefix slope, prefix GR calibration. | Never backfill into the hidden tail. |
| Typewell `TVT -> GR` | Reference curve for alignment. | A good GR match is not automatically the correct datum. |
| Train formation columns | Diagnostics and fold-safe imputation ideas. | Not directly present on hidden test horizontal rows. |
| Hidden-tail `TVT` | Train labels and EDA diagnostics only. | Never a feature or validation shortcut. |

> **Rule of thumb.** Use every target-free geological signal, but never transform hidden-tail target values into features, weights, or validation shortcuts.


## 1.1 Strict/offline policy and overlap risk

> **Why this matters.** A feature can be target-free and still have different public/private behavior.

The old EDA notebook used a two-track information model.  This working note keeps the same distinction because it prevents two different mistakes: rejecting legal batch covariates as if they were leakage, and accepting overlap-specific shortcuts as if they were universally robust.

| Policy | Allowed inputs | Typical use | Main risk |
|---|---|---|---|
| Strict drilling-time | prefix rows and current/trailing row evidence | conservative validation, causal sanity checks | may underuse batch-visible hidden GR/trajectory |
| Offline batch | all provided test-file covariates except hidden `TVT` | Kaggle inference, PF/beam/full-tail diagnostics | can over-trust future GR or overlap structure |
| Public-aggressive overlap | target-free signals that depend on train/test well overlap | public reproduction and stress tests | may not transfer to unseen private wells |

A compact leakage rule is:

$$
\text{future GR / trajectory} \neq \text{future TVT}.
$$

Future GR and trajectory are provided covariates in the test CSVs.  Future hidden `TVT` is the answer key.  The notebook treats those as completely different objects.


Same-well physical/contact estimates are not part of the private-safe target-free feature set.  They are a separate public-aggressive overlap policy.  They may be valid for public reproduction, but they should not be used as evidence of unseen-well robustness.

**Self-verifying override rule.** The public-aggressive contact shortcut should not be a blind rule such as `if well_id in train: use contact_formula`.  The safer public-profile rule is `if well_id in train and the contact path verifies on the current test prefix: override`.  Verification must use the active test file's known `TVT_input` prefix and MD interpolation, not row-index alignment.  This keeps public overlap exploitation separated from private-safe target-free inference.


In [ ]:
show_public_figure('ROGII_Graph_Fig8.png')

**Figure. Public-aggressive versus private-safe feature policies.**  Target-free PF and beam tracking use observed logs and typewell information.  Same-well physical/contact estimates can exploit train/test overlap, so they belong to a separate policy bucket even when they do not directly read hidden-tail `TVT`.

### Feature policy boundary

The table below makes the information boundary operational.  A signal can be legal, useful, and still belong to a different validation bucket.  This is the main reason the notebook separates private-safe target-free estimators from public-aggressive overlap policies.


In [ ]:
feature_policy_table = pd.DataFrame([
    {
        'component': 'last-known TVT anchor',
        'uses_hidden_target': False,
        'uses_train_target': False,
        'uses_test_covariates': True,
        'policy': 'private-safe target-free',
        'note': 'Known prefix is part of the test input contract.',
    },
    {
        'component': 'GR interpolation and trajectory geometry',
        'uses_hidden_target': False,
        'uses_train_target': False,
        'uses_test_covariates': True,
        'policy': 'private-safe target-free',
        'note': 'Future GR/geometry are observed covariates, not hidden TVT.',
    },
    {
        'component': 'PF / beam using typewell GR',
        'uses_hidden_target': False,
        'uses_train_target': False,
        'uses_test_covariates': True,
        'policy': 'private-safe target-free',
        'note': 'Uses observed GR, typewell curves, trajectory, and prefix anchor.',
    },
    {
        'component': 'formation surface / spatial imputer',
        'uses_hidden_target': False,
        'uses_train_target': True,
        'uses_test_covariates': True,
        'policy': 'fold-safe train-derived',
        'note': 'OOF validation must rebuild surfaces without validation wells.',
    },
    {
        'component': 'same-well physical contact formula',
        'uses_hidden_target': False,
        'uses_train_target': True,
        'uses_test_covariates': True,
        'policy': 'public-aggressive overlap',
        'note': 'Allowed as a public profile only when verified on the current prefix.',
    },
    {
        'component': 'raw same-well formation/contact columns',
        'uses_hidden_target': False,
        'uses_train_target': True,
        'uses_test_covariates': True,
        'policy': 'public-aggressive overlap',
        'note': 'Powerful when public test wells overlap train wells; not unseen-well robustness evidence.',
    },
    {
        'component': 'public test prediction CSV',
        'uses_hidden_target': False,
        'uses_train_target': None,
        'uses_test_covariates': True,
        'policy': 'public-only, hidden-unsafe',
        'note': 'Hidden rerun sample ids may differ, so it is not a submission artifact.',
    },
])

display(feature_policy_table)


## 1.2 Artifact and feature-cache policy

> **Takeaway.** Train artifacts can be prepared in advance; hidden-test features and predictions must be rebuilt from the active test folder at rerun time.

This distinction matters because a model package is valid only if it stores reusable estimators, blend parameters, or train-derived references.  It should not store predictions for a specific public test snapshot and treat them as hidden-test evidence.  The executable table below is the operational boundary used throughout the note.

The train feature cache is a computational cache, not an additional raw dataset.  It is valid only because it is derived from the fixed official train folder.  The active hidden-test feature table must still be rebuilt inside the submission notebook.


In [ ]:
artifact_boundary = pd.DataFrame([
    {
        'artifact': 'train feature cache',
        'can_precompute': True,
        'hidden_safe': True,
        'reason': 'Official train data is fixed.',
        'use': 'training, OOF, schema checks',
    },
    {
        'artifact': 'saved fold/all-train models',
        'can_precompute': True,
        'hidden_safe': True,
        'reason': 'Models are reusable estimators applied to active hidden features.',
        'use': 'hidden inference',
    },
    {
        'artifact': 'blend weights / ridge coefficients',
        'can_precompute': True,
        'hidden_safe': True,
        'reason': 'OOF-fitted parameters, not test predictions.',
        'use': 'stacking and gated correction',
    },
    {
        'artifact': 'public test feature cache',
        'can_precompute': True,
        'hidden_safe': False,
        'reason': 'Public test ids may differ from the hidden rerun.',
        'use': 'public probe only',
    },
    {
        'artifact': 'public submission CSV',
        'can_precompute': True,
        'hidden_safe': False,
        'reason': 'Cannot cover active hidden sample_submission ids.',
        'use': 'public sanity check only',
    },
    {
        'artifact': 'hidden test feature cache',
        'can_precompute': False,
        'hidden_safe': 'build during rerun',
        'reason': 'Hidden test folder is only visible during the submission run.',
        'use': 'final notebook computation',
    },
])

display(artifact_boundary)


## 2. Data contract and visibility

> **Takeaway.** The core sample is a well tail, not an independent row.

Each horizontal well has a known prefix, then one hidden tail block.  The public input exposes the trajectory and GR log across the whole horizontal well, but the target coordinate `TVT` disappears after the prefix.

For a horizontal well $w$, define

$$
\mathcal{P}_w = \{i: T^{input}_{w,i}\ \text{is observed}\},
\qquad
\mathcal{H}_w = \{i: T^{input}_{w,i}\ \text{is missing}\}.
$$

Only rows in $\mathcal{H}_w$ enter the submission.  Rows in $\mathcal{P}_w$ are legal anchors for datum, slope, and GR/typewell compatibility checks.


**Figure A. ROGII competition data contract and prediction target.**  
Each well consists of a horizontal-well file and a typewell reference file. The horizontal well contains an observed prefix with `TVT_input` and a hidden prediction tail. The submission requires TVT predictions only for the hidden-tail sample rows, so the task is to recover the hidden stratigraphic path from the prefix, trajectory, GR log, and typewell reference.

In [ ]:
show_public_figure('ROGII_WorkingNote_FIG_A.png')

<!-- FIG_A_READING_GUIDE -->
**How to read Figure A.**  For each well $w$, the public object naturally splits into a visible prefix and hidden tail:

$$
\mathcal P_w=\{i:T^{input}_{w,i}\ \text{is observed}\},\qquad
\mathcal H_w=\{i:T^{input}_{w,i}\ \text{is missing}\}.
$$

The legal estimator can use the full observed covariate traces, the prefix target anchor, and the typewell reference,

$$
\hat T_{w,\mathcal H}
=F\left(X_{w,\mathcal P\cup\mathcal H},\ T^{input}_{w,\mathcal P},\ \text{typewell}_w\right),
$$

but it may not use $T_{w,\mathcal H}$ while building the prediction.  This distinction is the reason the note separates target-free diagnostics from train-only oracle measurements.

## 2.1 File inventory and submission mapping

The file structure encodes the modeling unit.

| Object | Role | Required join key |
|---|---|---|
| horizontal well CSV | observed trajectory, GR, prefix `TVT_input`, train tail labels | `well_id`, row index |
| typewell CSV | reference `TVT -> GR` curve | `well_id` |
| sample submission | exact hidden-tail rows to predict | `id = {well_id}_{row_index}` |
| optional figures/reports | explanation only | none |

The submission contract is simple but unforgiving:

$$
\text{submission rows} = \{(w,i): i \in \mathcal{H}_w\},
\qquad
\text{id}_{w,i}=\texttt{well\_id}\_i.
$$

That is why every diagnostic in this note is organized around whole wells and hidden-tail row sets.  A row can be evaluated only in the context of its prefix, typewell, and local trajectory.


In [ ]:

def well_id_from_path(path: Path) -> str:
    return path.name.split('__')[0]

def horizontal_files(split_dir: Path):
    return sorted(split_dir.glob('*__horizontal_well.csv'))

def typewell_files(split_dir: Path):
    return sorted(split_dir.glob('*__typewell.csv'))

train_horizontal_files = horizontal_files(TRAIN_DIR)
train_typewell_files = typewell_files(TRAIN_DIR)
test_horizontal_files = horizontal_files(TEST_DIR)
test_typewell_files = typewell_files(TEST_DIR)

sample_path = DATA_ROOT / 'sample_submission.csv'
if not sample_path.exists():
    sample_path = LOCAL_REPO_ROOT / 'sample_submission.csv'
sample_rows = len(pd.read_csv(sample_path, usecols=['id'])) if sample_path.exists() else np.nan

inventory = pd.DataFrame([
    {'split': 'train', 'horizontal_wells': len(train_horizontal_files), 'typewells': len(train_typewell_files), 'submission_rows': np.nan},
    {'split': 'test', 'horizontal_wells': len(test_horizontal_files), 'typewells': len(test_typewell_files), 'submission_rows': sample_rows},
])

display(inventory)

train_columns = pd.read_csv(train_horizontal_files[0], nrows=0).columns.tolist() if train_horizontal_files else []
test_columns = pd.read_csv(test_horizontal_files[0], nrows=0).columns.tolist() if test_horizontal_files else []
column_roles = pd.DataFrame({
    'column': sorted(set(train_columns) | set(test_columns)),
})
column_roles['in_train'] = column_roles['column'].isin(train_columns)
column_roles['in_test'] = column_roles['column'].isin(test_columns)
column_roles['role'] = np.select(
    [
        column_roles['column'].isin(['MD', 'X', 'Y', 'Z', 'GR']),
        column_roles['column'].eq('TVT_input'),
        column_roles['column'].eq('TVT'),
        column_roles['column'].isin(['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']),
    ],
    [
        'observable row covariate',
        'known-prefix target anchor',
        'train label / hidden target',
        'train-only formation diagnostic',
    ],
    default='other',
)
display(column_roles)


> **Why it matters.** A strong solution has two responsibilities: respect the known-prefix anchors, and infer the hidden-tail shape from noisy spatial and GR evidence.  The next figure is the operational form of that contract.


In [ ]:
show_public_figure('ROGII_Graph_Fig2.png')

## 3. Prediction-zone anatomy

> **Takeaway.** A small well-level offset or slope error can dominate thousands of rows.

For each train well we record prefix length, tail length, GR missingness, tail TVT range, and the constant-anchor baseline

$$
\hat T_{w,i}^{const} = T^{input}_{w,L(w)},
\qquad i \in \mathcal{H}_w,
$$

where $L(w)$ is the last known prefix row.

This baseline is intentionally simple.  It is the score scale before any alignment, geometry, or shape modeling.


## 3.1 Row-weighted metric and well-level failure modes

The leaderboard metric is row-weighted RMSE:

$$
RMSE_{row}=\sqrt{\frac{1}{\sum_w n_w}\sum_w\sum_{i\in\mathcal{H}_w} e_{w,i}^2}.
$$

A complementary well-level view is

$$
RMSE_{well}=\frac{1}{W}\sum_w\sqrt{\frac{1}{n_w}\sum_{i\in\mathcal{H}_w}e_{w,i}^2}.
$$

The two are not interchangeable.  A model can improve row RMSE by helping a few long wells while hurting many short wells.  Conversely, a well-level diagnostic can reveal frequent small failures that barely move the public score.

This is why the old EDA notebook tracked both:

| Diagnostic | Reads as |
|---|---|
| tail length | row-weight leverage of each well |
| tail TVT range | maximum possible datum/shape movement |
| constant-anchor RMSE | difficulty before geologic modeling |
| prefix length | strength of legal anchor evidence |
| hidden GR missingness | reliability of GR-based alignment |


In [ ]:

def _safe_rmse(y, pred):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    mask = np.isfinite(y) & np.isfinite(pred)
    if mask.sum() == 0:
        return np.nan
    return float(np.sqrt(np.mean((y[mask] - pred[mask]) ** 2)))

def summarize_horizontal_file(path: Path, split: str) -> dict:
    wid = well_id_from_path(path)
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    usecols = [c for c in ['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input', 'TVT'] if c in cols]
    df = pd.read_csv(path, usecols=usecols)
    hidden = df['TVT_input'].isna() if 'TVT_input' in df else pd.Series(False, index=df.index)
    prefix = ~hidden
    hidden_idx = np.flatnonzero(hidden.to_numpy())
    first_hidden = int(hidden_idx[0]) if len(hidden_idx) else len(df)
    last_known = int(max(first_hidden - 1, 0))

    out = {
        'split': split,
        'well_id': wid,
        'n_rows': int(len(df)),
        'known_rows': int(prefix.sum()),
        'tail_rows': int(hidden.sum()),
        'tail_fraction': float(hidden.mean()),
        'first_hidden_index': first_hidden,
        'gr_missing_rate': float(df['GR'].isna().mean()) if 'GR' in df else np.nan,
        'gr_missing_prefix_rate': float(df.loc[prefix, 'GR'].isna().mean()) if 'GR' in df and prefix.any() else np.nan,
        'gr_missing_tail_rate': float(df.loc[hidden, 'GR'].isna().mean()) if 'GR' in df and hidden.any() else np.nan,
    }

    for axis in ['X', 'Y', 'Z', 'MD']:
        if axis in df:
            vals = df[axis].to_numpy(dtype=float)
            out[f'{axis.lower()}_span'] = float(np.nanmax(vals) - np.nanmin(vals)) if np.isfinite(vals).any() else np.nan

    if {'X', 'Y'}.issubset(df.columns):
        x = df['X'].to_numpy(dtype=float)
        y = df['Y'].to_numpy(dtype=float)
        out['xy_span'] = float(np.hypot(np.nanmax(x) - np.nanmin(x), np.nanmax(y) - np.nanmin(y)))
        if 0 <= last_known < len(df):
            out['ps_x'] = float(df.loc[last_known, 'X'])
            out['ps_y'] = float(df.loc[last_known, 'Y'])
            if len(df) > 1:
                dx = float(df['X'].iloc[-1] - df['X'].iloc[0])
                dy = float(df['Y'].iloc[-1] - df['Y'].iloc[0])
                out['azimuth_deg'] = float(np.degrees(np.arctan2(dy, dx)))

    if split == 'train' and 'TVT' in df and hidden.any() and prefix.any():
        tail_y = df.loc[hidden, 'TVT'].to_numpy(dtype=float)
        last_anchor = float(df.loc[prefix, 'TVT_input'].dropna().iloc[-1])
        out.update({
            'last_known_tvt': last_anchor,
            'tail_tvt_range': float(np.nanmax(tail_y) - np.nanmin(tail_y)),
            'tail_end_delta_from_last_known': float(tail_y[-1] - last_anchor),
            'tail_median_abs_step': float(np.nanmedian(np.abs(np.diff(tail_y)))) if len(tail_y) > 1 else np.nan,
            'constant_tail_rmse': _safe_rmse(tail_y, np.full_like(tail_y, last_anchor, dtype=float)),
        })
    return out

well_summary = pd.DataFrame([summarize_horizontal_file(p, 'train') for p in train_horizontal_files])
test_well_summary = pd.DataFrame([summarize_horizontal_file(p, 'test') for p in test_horizontal_files])
well_summary.to_csv(OUT_DIR / 'well_summary.csv', index=False)
test_well_summary.to_csv(OUT_DIR / 'test_well_summary.csv', index=False)

summary_cols = [
    'n_rows', 'known_rows', 'tail_rows', 'tail_fraction',
    'gr_missing_rate', 'gr_missing_prefix_rate', 'gr_missing_tail_rate',
    'tail_tvt_range', 'tail_end_delta_from_last_known', 'tail_median_abs_step',
    'constant_tail_rmse', 'xy_span', 'z_span', 'md_span',
]
existing_summary_cols = [c for c in summary_cols if c in well_summary.columns]
display(well_summary[existing_summary_cols].describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]).T.round(3))

plot_specs = [
    ('n_rows', 'Rows per well'),
    ('known_rows', 'Known-prefix rows'),
    ('tail_rows', 'Hidden-tail rows'),
    ('gr_missing_rate', 'GR missing rate'),
    ('tail_tvt_range', 'Hidden-tail TVT range'),
    ('constant_tail_rmse', 'Constant-anchor RMSE'),
]
fig, axes = plt.subplots(2, 3, figsize=(14, 7.8), constrained_layout=True)
for ax, (col, title) in zip(axes.ravel(), plot_specs):
    vals = well_summary[col].replace([np.inf, -np.inf], np.nan).dropna()
    sns.histplot(vals, bins=min(36, max(8, int(np.sqrt(max(len(vals), 1))))), ax=ax, color='#3d6fb6', edgecolor='white')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(col)
    ax.set_ylabel('well count')
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x:.0f}' if abs(x) >= 10 else f'{x:.2g}'))
    ax.grid(True, alpha=0.25)
fig.suptitle('Well-level prediction-zone anatomy', fontsize=15)
fig.savefig(OUT_DIR / 'fig_well_summary.png', dpi=160, bbox_inches='tight')
plt.show()


> **Reading guide.** The summary is not just descriptive.  It tells us whether the score is driven by long-tail well failures, prefix-anchor offsets, or GR/shape issues.  If a method cannot beat the constant anchor, it is probably spending complexity on the wrong part of the problem.


## 4. Geometry: TVT as a surface coordinate

> **Takeaway.** TVT behaves like a vertical coordinate relative to a formation surface.

For a horizontal well, a useful approximation is

$$
T_w(m) = S_w(m) - Z_w(m),
$$

or, equivalently,

$$
T_w(m) + Z_w(m) = S_w(m).
$$

The evaluation rows already contain $MD, X, Y, Z, GR$.  What is hidden is the vertical stratigraphic position, $T$.  The typewell supplies a GR-vs-TVT reference curve, so the horizontal GR trace acts as a stratigraphic ruler.  It is informative, but not perfect.


## 4.1 Path derivatives and curve geometry

A horizontal well is a path through a structural volume.  The old EDA used simple trajectory proxies to make this visible:

$$
\frac{dZ}{dMD}_i = \frac{Z_i-Z_{i-1}}{MD_i-MD_{i-1}},
\qquad
 dXY_i=\sqrt{(X_i-X_{i-1})^2+(Y_i-Y_{i-1})^2}.
$$

A lightweight curvature proxy is the local change in normalized path direction:

$$
\kappa_i \approx
\sqrt{\Delta(dX/dMD)^2+\Delta(dY/dMD)^2+\Delta(dZ/dMD)^2}.
$$

These are not magic features by themselves.  Their job is to tell whether the hidden tail is mostly flat, gradually dipping, or passing through a path segment where slope transfer is likely to break.


> **Caution.** The hidden rows contain the full horizontal trajectory and GR log, so target-free within-well GR matching is allowed.  What is not allowed is using future hidden TVT labels.  This is a post-drilling alignment task, not a sequential forecast with hidden future covariates.


In [ ]:
show_public_figure('ROGII_Graph_Fig1.png')


## 5. TVT_input consistency and target behavior

> **Takeaway.** The prefix is a clean legal anchor, but the tail still needs curve-level reasoning.

Two checks matter before modeling:

| Check | Why |
|---|---|
| Does `TVT_input` equal train `TVT` on known rows? | If yes, `last_known_TVT`, prefix slope, and prefix range are legal quantities. |
| Is tail TVT smooth enough to model as a curve? | Smoothness supports shrinkage and curve diagnostics, but non-monotonic behavior warns against blind extrapolation. |


In [ ]:

prefix_consistency_rows = []
dtvt_chunks = []
well_target_rows = []
for path in train_horizontal_files:
    wid = well_id_from_path(path)
    df = pd.read_csv(path, usecols=['MD', 'TVT', 'TVT_input'])
    known = df['TVT_input'].notna()
    if known.any():
        err = (df.loc[known, 'TVT'] - df.loc[known, 'TVT_input']).astype(float)
        prefix_consistency_rows.append({
            'well_id': wid,
            'known_rows': int(known.sum()),
            'max_abs_prefix_error': float(err.abs().max()),
            'mean_abs_prefix_error': float(err.abs().mean()),
        })
    tvt = pd.to_numeric(df['TVT'], errors='coerce').to_numpy(dtype=float)
    d = np.diff(tvt)
    d = d[np.isfinite(d)]
    if len(d):
        dtvt_chunks.append(d)
        well_target_rows.append({
            'well_id': wid,
            'dtvt_median': float(np.nanmedian(d)),
            'dtvt_median_abs': float(np.nanmedian(np.abs(d))),
            'dtvt_p95_abs': float(np.nanpercentile(np.abs(d), 95)),
            'dtvt_positive_rate': float(np.mean(d > 0)),
            'dtvt_negative_rate': float(np.mean(d < 0)),
        })

prefix_consistency = pd.DataFrame(prefix_consistency_rows)
target_step_summary = pd.DataFrame(well_target_rows)
dtvt_values = np.concatenate(dtvt_chunks) if dtvt_chunks else np.array([], dtype=float)
prefix_consistency.to_csv(OUT_DIR / 'prefix_tvt_input_consistency.csv', index=False)
target_step_summary.to_csv(OUT_DIR / 'target_step_summary_by_well.csv', index=False)

display(pd.DataFrame([
    {'metric': 'wells_checked', 'value': len(prefix_consistency)},
    {'metric': 'max_prefix_abs_error_over_all_wells', 'value': float(prefix_consistency['max_abs_prefix_error'].max())},
    {'metric': 'wells_with_nonzero_prefix_error', 'value': int((prefix_consistency['max_abs_prefix_error'] > 1e-9).sum())},
    {'metric': 'row_dTVT_median', 'value': float(np.nanmedian(dtvt_values))},
    {'metric': 'row_abs_dTVT_median', 'value': float(np.nanmedian(np.abs(dtvt_values)))},
    {'metric': 'row_abs_dTVT_p95', 'value': float(np.nanpercentile(np.abs(dtvt_values), 95))},
]).round(6))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)
clipped = dtvt_values[np.isfinite(dtvt_values)]
clipped = clipped[np.abs(clipped) <= np.nanpercentile(np.abs(clipped), 99.5)]
sns.histplot(clipped, bins=80, ax=axes[0], color='#4e79a7', edgecolor='white')
axes[0].axvline(0, color='black', lw=1.0, ls='--')
axes[0].set_title('Row-level dTVT distribution')
axes[0].set_xlabel('TVT[i] - TVT[i-1]')
axes[0].set_ylabel('row count')

sns.histplot(target_step_summary['dtvt_median_abs'].dropna(), bins=36, ax=axes[1], color='#59a14f', edgecolor='white')
axes[1].set_title('Per-well median |dTVT|')
axes[1].set_xlabel('median absolute step')
axes[1].set_ylabel('well count')
axes[1].yaxis.set_major_locator(MaxNLocator(integer=True))

sns.histplot(target_step_summary['dtvt_p95_abs'].dropna(), bins=36, ax=axes[2], color='#f28e2b', edgecolor='white')
axes[2].set_title('Per-well 95th percentile |dTVT|')
axes[2].set_xlabel('p95 absolute step')
axes[2].set_ylabel('well count')
axes[2].yaxis.set_major_locator(MaxNLocator(integer=True))
for ax in axes:
    ax.grid(True, alpha=0.25)
fig.suptitle('TVT prefix consistency and target smoothness', fontsize=15)
fig.savefig(OUT_DIR / 'fig_tvt_consistency_and_smoothness.png', dpi=160, bbox_inches='tight')
plt.show()


## 6. Baseline evaluation before modeling

> **Takeaway.** Null models define the burden of proof.

The first null model is the constant anchor:

$$
\hat T_{w,i}^{const}=T^{input}_{w,L(w)}.
$$

The second is a clipped prefix-slope extrapolation through the last known anchor:

$$
\hat T_{w,i}^{slope}=T^{input}_{w,L(w)}+\hat\beta_w\,(MD_{w,i}-MD_{w,L(w)}).
$$

These are not final methods.  They tell us what must be beaten and where simple extrapolation fails.

**Diagnostic conclusion.**  The anchor is the minimum credible baseline; prefix slope is a diagnostic, not a global predictor.

**Modeling action.**  Train residuals around the anchor and use slope only as a clipped or gated feature.

**Policy.**  Private-safe target-free, because it uses only the known prefix.

**Guardrail.**  Any proposed correction must beat the anchor in row RMSE and avoid increasing the number of badly hurt wells.


In [ ]:

BASELINE_PREFIX_WINDOW = 120
BASELINE_SLOPE_CLIP = 0.05


def _baseline_rmse_sse(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    m = np.isfinite(y) & np.isfinite(p)
    if int(m.sum()) == 0:
        return np.nan, 0, 0.0
    e = y[m] - p[m]
    return float(np.sqrt(np.mean(e * e))), int(m.sum()), float(np.sum(e * e))


def _prefix_slope_from_tail(md_known, tvt_known, window=BASELINE_PREFIX_WINDOW, slope_clip=BASELINE_SLOPE_CLIP):
    md_known = np.asarray(md_known, dtype=float)
    tvt_known = np.asarray(tvt_known, dtype=float)
    m = np.isfinite(md_known) & np.isfinite(tvt_known)
    md_known, tvt_known = md_known[m], tvt_known[m]
    if len(md_known) < 5:
        return np.nan, np.nan
    order = np.argsort(md_known)
    md_known, tvt_known = md_known[order], tvt_known[order]
    k = min(window, len(md_known))
    md_known = md_known[-k:]
    tvt_known = tvt_known[-k:]
    dmd = np.diff(md_known)
    dtvt = np.diff(tvt_known)
    ok = np.isfinite(dmd) & np.isfinite(dtvt) & (np.abs(dmd) > 1e-12)
    rates = dtvt[ok] / dmd[ok]
    rates = rates[np.isfinite(rates)]
    if len(rates) == 0:
        return np.nan, np.nan
    raw = float(np.nanmedian(rates))
    clipped = float(np.clip(raw, -slope_clip, slope_clip))
    return raw, clipped


def baseline_eval_for_well(path):
    wid = well_id_from_path(path)
    df = pd.read_csv(path, usecols=['MD', 'TVT', 'TVT_input'])
    known = df['TVT_input'].notna().to_numpy()
    hidden = ~known
    if known.sum() < 5 or hidden.sum() == 0:
        return None
    mdv = pd.to_numeric(df['MD'], errors='coerce').to_numpy(dtype=float)
    y = pd.to_numeric(df['TVT'], errors='coerce').to_numpy(dtype=float)
    tvt_input = pd.to_numeric(df['TVT_input'], errors='coerce').to_numpy(dtype=float)
    last_idx = np.flatnonzero(known)[-1]
    eval_idx = np.flatnonzero(hidden)
    y_tail = y[eval_idx]
    md_tail = mdv[eval_idx]
    last_tvt = float(tvt_input[last_idx])
    last_md = float(mdv[last_idx])
    raw_slope, clipped_slope = _prefix_slope_from_tail(mdv[known], tvt_input[known])
    const_pred = np.full(len(eval_idx), last_tvt, dtype=float)
    clipped_pred = last_tvt + clipped_slope * (md_tail - last_md) if np.isfinite(clipped_slope) else const_pred.copy()
    oracle_mean_pred = np.full(len(eval_idx), float(np.nanmean(y_tail)), dtype=float)
    out = {'well_id': wid, 'rows': int(len(eval_idx)), 'prefix_slope_raw': raw_slope, 'prefix_slope_clipped': clipped_slope}
    for name, pred in [('constant_anchor', const_pred), ('clipped_prefix_slope', clipped_pred), ('oracle_tail_mean', oracle_mean_pred)]:
        r, n, sse = _baseline_rmse_sse(y_tail, pred)
        out[f'{name}_rmse'] = r
        out[f'{name}_rows'] = n
        out[f'{name}_sse'] = sse
    return out

baseline_rows = [baseline_eval_for_well(p) for p in train_horizontal_files]
baseline_rows = [r for r in baseline_rows if r is not None]
baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(OUT_DIR / 'baseline_eval_by_well.csv', index=False)

baseline_summary_rows = []
for name in ['constant_anchor', 'clipped_prefix_slope', 'oracle_tail_mean']:
    sse = float(baseline_df[f'{name}_sse'].sum())
    n = int(baseline_df[f'{name}_rows'].sum())
    baseline_summary_rows.append({'model': name, 'pooled_rmse': math.sqrt(sse / n), 'rows': n, 'wells': len(baseline_df)})
baseline_summary = pd.DataFrame(baseline_summary_rows)
baseline_summary.to_csv(OUT_DIR / 'baseline_eval_summary.csv', index=False)
display(baseline_summary.round(3))

fig, ax = plt.subplots(figsize=(8.4, 4.4))
colors = ['#5B7CFA', '#E8743B', '#8A8F98']
ax.bar(baseline_summary['model'], baseline_summary['pooled_rmse'], color=colors)
for x, yv in enumerate(baseline_summary['pooled_rmse']):
    ax.text(x, yv + 0.3, f'{yv:.2f}', ha='center', va='bottom', fontsize=11)
ax.set_title('Simple train-side tail baselines')
ax.set_ylabel('pooled RMSE')
ax.set_xlabel('baseline')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig_baseline_eval.png', dpi=160, bbox_inches='tight')
plt.show()


> **Reading guide.** The constant anchor is a serious baseline.  A good method must improve on it without importing unstable prefix-slope behavior into wells where the tail changes regime.  The oracle tail mean is train-only; it only shows the size of a per-well datum/offset gap.


## 7. GR/typewell alignment signal

> **Takeaway.** GR is a likelihood signal, not an oracle.

The known prefix lets us measure whether the horizontal GR curve is compatible with the typewell GR curve at the known `TVT_input` coordinates.  A simple prefix residual is

$$
r_{w,i} = GR^{horizontal}_{w,i} - GR^{typewell}_w\!\left(T^{input}_{w,i}\right),
\qquad i \in \mathcal{P}_w.
$$

The residual scale is a practical observation-noise estimate:

| Prefix residual | Interpretation |
|---|---|
| Low | Typewell alignment has a chance. |
| High | GR matching should be down-weighted or hedged. |

**Diagnostic conclusion.**  GR/typewell matching is informative, but the residual scale is well-specific.

**Modeling action.**  Use GR as an observation likelihood inside PF, beam search, and posterior averaging.

**Policy.**  Private-safe target-free, because it uses observed horizontal GR, typewell GR, and the known prefix anchor.

**Guardrail.**  Down-weight GR likelihood when prefix residuals are large or missingness is high.


## 7.1 GR quality, gaps, and observation likelihood

GR is the main stratigraphic barcode, but missing intervals change how much the barcode can be trusted.

| GR condition | Modeling effect |
|---|---|
| short isolated gaps | interpolation is usually harmless |
| long hidden gaps | PF/beam likelihood weakens |
| high prefix missingness | prefix calibration scale is unstable |
| high hidden missingness | geometry and formation priors should carry more weight |

The prefix residual scale also defines a natural observation-noise proxy:

$$
gs_w = \operatorname{std}\left(GR^{horizontal}_{w,i} - GR^{typewell}_w(T^{input}_{w,i})\right),
\qquad i\in\mathcal{P}_w.
$$

Low $gs_w$ means typewell tracking can be sharp.  High $gs_w$ means the same likelihood should be flatter and more hedge-friendly.


In [ ]:
show_public_figure('ROGII_Graph_Fig4.png')

**Figure. Gamma ray as a stratigraphic barcode.**  Hidden horizontal GR is aligned against typewell GR to infer stratigraphic position.  Missing GR gaps are interpolated before PF/beam tracking so that the observation likelihood remains continuous.

In [ ]:

def _interp_typewell_gr(typewell_df: pd.DataFrame, tvt_values: np.ndarray) -> np.ndarray:
    tw = typewell_df[['TVT', 'GR']].dropna().sort_values('TVT')
    if len(tw) < 2:
        return np.full(len(tvt_values), np.nan)
    x = tw['TVT'].to_numpy(dtype=float)
    y = tw['GR'].to_numpy(dtype=float)
    tvt_values = np.asarray(tvt_values, dtype=float)
    pred = np.interp(tvt_values, x, y)
    pred[(tvt_values < x[0]) | (tvt_values > x[-1])] = np.nan
    return pred

def _safe_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 4:
        return np.nan
    aa = a[mask]
    bb = b[mask]
    if np.nanstd(aa) == 0 or np.nanstd(bb) == 0:
        return np.nan
    return float(np.corrcoef(aa, bb)[0, 1])

def summarize_prefix_typewell_alignment(wid: str) -> dict:
    h_path = TRAIN_DIR / f'{wid}__horizontal_well.csv'
    tw_path = TRAIN_DIR / f'{wid}__typewell.csv'
    h = pd.read_csv(h_path, usecols=['GR', 'TVT_input', 'TVT'])
    tw = pd.read_csv(tw_path, usecols=['TVT', 'GR'])
    known = h['TVT_input'].notna()
    obs = h.loc[known, 'GR'].to_numpy(dtype=float)
    tvt = h.loc[known, 'TVT_input'].to_numpy(dtype=float)
    ref = _interp_typewell_gr(tw, tvt)
    valid = np.isfinite(obs) & np.isfinite(ref)
    resid = obs[valid] - ref[valid]
    return {
        'well_id': wid,
        'prefix_rows': int(known.sum()),
        'valid_gr_pairs': int(valid.sum()),
        'prefix_pair_fraction': float(valid.sum() / max(known.sum(), 1)),
        'prefix_tw_gr_rmse': _safe_rmse(obs[valid], ref[valid]) if valid.any() else np.nan,
        'prefix_tw_gr_mae': float(np.nanmean(np.abs(resid))) if len(resid) else np.nan,
        'prefix_tw_gr_bias': float(np.nanmean(resid)) if len(resid) else np.nan,
        'prefix_tw_gr_resid_std': float(np.nanstd(resid)) if len(resid) else np.nan,
        'prefix_horizontal_vs_typewell_gr_corr': _safe_corr(obs, ref),
    }

alignment_summary = pd.DataFrame([
    summarize_prefix_typewell_alignment(well_id_from_path(p)) for p in train_horizontal_files
])
alignment_summary = alignment_summary.merge(
    well_summary[['well_id', 'constant_tail_rmse', 'tail_rows', 'gr_missing_rate']],
    on='well_id',
    how='left',
)
alignment_summary.to_csv(OUT_DIR / 'prefix_typewell_alignment_summary.csv', index=False)

display(alignment_summary.describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]).T.round(3))

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), constrained_layout=True)
plot_cols = [
    ('prefix_tw_gr_rmse', 'Prefix GR/typewell RMSE'),
    ('prefix_tw_gr_resid_std', 'Prefix GR residual std'),
    ('prefix_horizontal_vs_typewell_gr_corr', 'Prefix GR correlation'),
]
for ax, (col, title) in zip(axes[:3], plot_cols):
    vals = alignment_summary[col].replace([np.inf, -np.inf], np.nan).dropna()
    sns.histplot(vals, bins=min(36, max(8, int(np.sqrt(max(len(vals), 1))))), ax=ax, color='#4f9a7a', edgecolor='white')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(col)
    ax.set_ylabel('well count')
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x:.2f}'))
    ax.grid(True, alpha=0.25)

scatter_df = alignment_summary[['prefix_tw_gr_rmse', 'constant_tail_rmse', 'tail_rows']].replace([np.inf, -np.inf], np.nan).dropna()
axes[3].scatter(
    scatter_df['prefix_tw_gr_rmse'],
    scatter_df['constant_tail_rmse'],
    s=np.clip(scatter_df['tail_rows'] / 35, 10, 90),
    alpha=0.45,
    color='#6b5fb5',
    edgecolor='none',
)
axes[3].set_title('Alignment quality vs tail difficulty', fontsize=12)
axes[3].set_xlabel('prefix_tw_gr_rmse')
axes[3].set_ylabel('constant_tail_rmse')
axes[3].grid(True, alpha=0.25)
fig.suptitle('Known-prefix GR/typewell compatibility diagnostics', fontsize=15)
fig.savefig(OUT_DIR / 'fig_prefix_typewell_alignment.png', dpi=160, bbox_inches='tight')
plt.show()


> **Why it matters.** A well can have a plausible typewell match and still lack enough information to choose between neighboring datum modes.  Conversely, when the prefix residual is poor, forcing a single GR-derived alignment can be worse than using a conservative datum and leaving shape uncertainty explicit.


## 8. Spatial and formation-surface signal

> **Takeaway.** Spatial geology is a prior, not a lookup table.

The field-level view is that TVT behaves like a structural surface observed along a horizontal trajectory.  Formation columns in train are valuable diagnostics because they show coherent spatial and formation-relative structure.

The conservative lesson is to use spatial context for datum and trend, while avoiding direct target lookup behavior.

**Diagnostic conclusion.**  $TVT+Z$ has structured relation to formation surfaces, but that structure is a prior rather than a row lookup.

**Modeling action.**  Use formation-surface residuals, dense ANCC, and PF-ANCC style state features.

**Policy.**  Fold-safe train-derived when train targets are used; target-free when derived only from visible/test covariates.

**Guardrail.**  Rebuild train-derived surfaces inside OOF folds and keep same-well contact logic in the public-aggressive bucket.


## 8.1 Formation-surface proxy

The useful physical approximation is

$$
TVT_i \approx -Z_i + \widehat S_F(X_i,Y_i) + b_{w,F},
$$

where $\widehat S_F(X,Y)$ is a spatial estimate of a formation surface and $b_{w,F}$ is a prefix-estimated well offset.

The important leakage distinction is:

| Use | Status |
|---|---|
| inspect train formation columns to understand surfaces | EDA diagnostic |
| fit a fold-safe spatial surface from training wells | target-free reference model |
| copy train-only formation columns into hidden test rows | invalid feature mismatch |
| use validation-well target surfaces inside a fold | fold leakage |

This is why formation logic is best treated as a smooth prior.  It can guide datum and trend, but it should not become a direct lookup table.


In [ ]:
show_public_figure('ROGII_Graph_Fig3.png')

## 9. Prefix GR shift landscape as a self-check

> **Takeaway.** GR argmin is usually informative, but a small number of multimodal/tie wells are high-impact enough that hard argmin selection is unsafe.

Ask a prefix-only question:

> If the true prefix coordinate is $T^{input}$, how often does pure GR matching prefer $T^{input}+\delta$?

For each well, scan shifts $\delta$ and compare horizontal prefix GR with the typewell GR sampled at $T^{input}+\delta$:

$$
C_w(\delta) =
\left[
\frac{1}{|\mathcal{P}_w|}
\sum_{i \in \mathcal{P}_w}
\left(GR^{h}_{w,i}-GR^{tw}_{w}(T^{input}_{w,i}+\delta)\right)^2
\right]^{1/2}.
$$

Here $\delta=0$ is the known-prefix TVT coordinate.  In the current recompute, most wells are close to centered, but a minority have shifted or secondary minima at bundle scale.  Those minority wells can still dominate leaderboard movement because a hard mode error is a whole-well datum error.

**Diagnostic conclusion.**  A shifted GR minimum is a warning flag, not a reason to force a shifted datum.

**Modeling action.**  Preserve multiple candidate paths and use likelihood spread as uncertainty.

**Policy.**  Private-safe target-free when computed from prefix and observed GR only.

**Guardrail.**  Do not collapse a broad or bimodal landscape into one hard argmin; report spread or hedge it.


In [ ]:

SHIFT_GRID = np.linspace(-40.0, 40.0, 321)
MAX_PREFIX_PAIRS_FOR_LANDSCAPE = 1400
MIN_SHIFT_VALID_PAIRS = 80
COMPETITIVE_COST_ABS = 0.50
COMPETITIVE_COST_REL = 0.03
SECONDARY_MIN_SEPARATION_FT = 8.0
BUNDLE_GAP_RANGE_FT = (8.0, 25.0)


def _interp_typewell_matrix(tw_tvt, tw_gr, tvt_matrix):
    flat = tvt_matrix.reshape(-1)
    pred = np.interp(flat, tw_tvt, tw_gr)
    pred[(flat < tw_tvt[0]) | (flat > tw_tvt[-1])] = np.nan
    return pred.reshape(tvt_matrix.shape)


def _local_minima_indices(cost):
    cost = np.asarray(cost, dtype=float)
    idx = []
    for i in range(1, len(cost) - 1):
        if np.isfinite(cost[i]) and cost[i] <= cost[i - 1] and cost[i] <= cost[i + 1]:
            idx.append(i)
    if np.isfinite(cost[0]) and cost[0] <= cost[1]:
        idx.append(0)
    if np.isfinite(cost[-1]) and cost[-1] <= cost[-2]:
        idx.append(len(cost) - 1)
    return np.asarray(idx, dtype=int)


def gr_shift_landscape_for_well(wid):
    h = pd.read_csv(TRAIN_DIR / f'{wid}__horizontal_well.csv', usecols=['GR', 'TVT_input'])
    tw = pd.read_csv(TRAIN_DIR / f'{wid}__typewell.csv', usecols=['TVT', 'GR']).dropna().sort_values('TVT')
    if len(tw) < 8:
        return None
    known = h['TVT_input'].notna()
    obs = h.loc[known, 'GR'].to_numpy(dtype=float)
    tvt = h.loc[known, 'TVT_input'].to_numpy(dtype=float)
    valid = np.isfinite(obs) & np.isfinite(tvt)
    obs, tvt = obs[valid], tvt[valid]
    if len(obs) < MIN_SHIFT_VALID_PAIRS:
        return None

    if len(obs) > MAX_PREFIX_PAIRS_FOR_LANDSCAPE:
        take = np.linspace(0, len(obs) - 1, MAX_PREFIX_PAIRS_FOR_LANDSCAPE).round().astype(int)
        obs = obs[take]
        tvt = tvt[take]

    tw_tvt = tw['TVT'].to_numpy(dtype=float)
    tw_gr = tw['GR'].to_numpy(dtype=float)
    tvt_shifted = tvt[:, None] + SHIFT_GRID[None, :]
    ref = _interp_typewell_matrix(tw_tvt, tw_gr, tvt_shifted)
    resid = obs[:, None] - ref
    ok = np.isfinite(resid)
    clipped = np.clip(resid, -75.0, 75.0)
    sse = np.nansum(np.where(ok, clipped * clipped, np.nan), axis=0)
    cnt = ok.sum(axis=0)
    cost = np.full(len(SHIFT_GRID), np.nan, dtype=float)
    good = cnt >= MIN_SHIFT_VALID_PAIRS
    cost[good] = np.sqrt(sse[good] / cnt[good])

    finite = np.flatnonzero(np.isfinite(cost))
    if len(finite) == 0:
        return None
    best_idx = int(finite[np.nanargmin(cost[finite])])
    zero_idx = int(np.argmin(np.abs(SHIFT_GRID)))
    best_delta = float(SHIFT_GRID[best_idx])
    best_cost = float(cost[best_idx])
    zero_cost = float(cost[zero_idx]) if np.isfinite(cost[zero_idx]) else np.nan
    zero_rank = int(1 + np.sum(cost[np.isfinite(cost)] < zero_cost)) if np.isfinite(zero_cost) else np.nan

    local_idx = _local_minima_indices(cost)
    local_idx = sorted(local_idx, key=lambda j: cost[j])
    second_idx = None
    for j in local_idx:
        if abs(float(SHIFT_GRID[j]) - best_delta) >= SECONDARY_MIN_SEPARATION_FT:
            second_idx = int(j)
            break
    second_delta = float(SHIFT_GRID[second_idx]) if second_idx is not None else np.nan
    second_cost = float(cost[second_idx]) if second_idx is not None else np.nan
    secondary_gap = abs(second_delta - best_delta) if np.isfinite(second_delta) else np.nan
    competitive_tol = max(COMPETITIVE_COST_ABS, COMPETITIVE_COST_REL * best_cost)
    competitive = np.isfinite(cost) & (cost <= best_cost + competitive_tol)

    return {
        'well_id': wid,
        'prefix_pairs_used': int(len(obs)),
        'best_delta_ft': best_delta,
        'best_abs_delta_ft': abs(best_delta),
        'best_cost': best_cost,
        'zero_cost': zero_cost,
        'zero_minus_best_cost': float(zero_cost - best_cost) if np.isfinite(zero_cost) else np.nan,
        'zero_rank': zero_rank,
        'local_minima_count': int(len(local_idx)),
        'competitive_shift_count': int(competitive.sum()),
        'competitive_delta_span_ft': float(np.nanmax(SHIFT_GRID[competitive]) - np.nanmin(SHIFT_GRID[competitive])) if competitive.any() else np.nan,
        'second_delta_ft': second_delta,
        'second_cost': second_cost,
        'second_minus_best_cost': float(second_cost - best_cost) if np.isfinite(second_cost) else np.nan,
        'secondary_gap_ft': secondary_gap,
        'has_bundle_scale_secondary': bool(
            np.isfinite(secondary_gap)
            and BUNDLE_GAP_RANGE_FT[0] <= secondary_gap <= BUNDLE_GAP_RANGE_FT[1]
            and np.isfinite(second_cost)
            and second_cost <= best_cost + max(1.0, 0.05 * best_cost)
        ),
        'best_shift_wrong_gt_2ft': bool(abs(best_delta) > 2.0),
        'best_shift_wrong_gt_5ft': bool(abs(best_delta) > 5.0),
        'best_shift_wrong_gt_10ft': bool(abs(best_delta) > 10.0),
    }

shift_rows = []
for wid in well_summary['well_id'].tolist():
    row = gr_shift_landscape_for_well(wid)
    if row is not None:
        shift_rows.append(row)

gr_shift_df = pd.DataFrame(shift_rows)
gr_shift_df.to_csv(OUT_DIR / 'prefix_gr_shift_landscape_summary.csv', index=False)

shift_report = pd.DataFrame([
    {'metric': 'wells_scanned', 'value': len(gr_shift_df)},
    {'metric': 'median_abs_best_shift_ft', 'value': gr_shift_df['best_abs_delta_ft'].median()},
    {'metric': 'pct_best_shift_abs_gt_2ft', 'value': 100.0 * gr_shift_df['best_shift_wrong_gt_2ft'].mean()},
    {'metric': 'pct_best_shift_abs_gt_5ft', 'value': 100.0 * gr_shift_df['best_shift_wrong_gt_5ft'].mean()},
    {'metric': 'pct_best_shift_abs_gt_10ft', 'value': 100.0 * gr_shift_df['best_shift_wrong_gt_10ft'].mean()},
    {'metric': 'median_zero_rank', 'value': gr_shift_df['zero_rank'].median()},
    {'metric': 'median_zero_minus_best_cost', 'value': gr_shift_df['zero_minus_best_cost'].median()},
    {'metric': 'pct_bundle_scale_secondary', 'value': 100.0 * gr_shift_df['has_bundle_scale_secondary'].mean()},
])
shift_report.to_csv(OUT_DIR / 'prefix_gr_shift_landscape_report.csv', index=False)
display(shift_report.round(3))

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.0), constrained_layout=True)

sns.histplot(gr_shift_df['best_delta_ft'].dropna(), bins=40, ax=axes[0, 0], color='#4e79a7', edgecolor='white')
axes[0, 0].axvline(0, color='black', lw=1.5, ls='--')
axes[0, 0].set_title('Best prefix GR shift')
axes[0, 0].set_xlabel('best_delta_ft; true prefix coordinate is 0')
axes[0, 0].set_ylabel('well count')
axes[0, 0].yaxis.set_major_locator(MaxNLocator(integer=True))

sns.histplot(gr_shift_df['zero_minus_best_cost'].dropna(), bins=36, ax=axes[0, 1], color='#f28e2b', edgecolor='white')
axes[0, 1].axvline(0, color='black', lw=1.5, ls='--')
axes[0, 1].set_title('GR-cost improvement over true prefix depth')
axes[0, 1].set_xlabel('C(0) - min_delta C(delta)')
axes[0, 1].set_ylabel('well count')
axes[0, 1].yaxis.set_major_locator(MaxNLocator(integer=True))

axes[1, 0].scatter(
    gr_shift_df['best_abs_delta_ft'],
    gr_shift_df['zero_minus_best_cost'],
    s=np.clip(gr_shift_df['prefix_pairs_used'] / 30, 10, 80),
    alpha=0.42,
    color='#59a14f',
    edgecolor='none',
)
axes[1, 0].axvline(5, color='black', lw=1.0, ls=':')
axes[1, 0].axhline(0, color='black', lw=1.0, ls='--')
axes[1, 0].set_title('Wrong-depth strength')
axes[1, 0].set_xlabel('|best_delta_ft|')
axes[1, 0].set_ylabel('C(0) - C(best)')

bundle = gr_shift_df['has_bundle_scale_secondary'].map({True: 'bundle-scale secondary', False: 'other'})
sns.scatterplot(
    data=gr_shift_df,
    x='best_delta_ft',
    y='second_delta_ft',
    hue=bundle,
    palette={'bundle-scale secondary': '#e15759', 'other': '#9c9c9c'},
    alpha=0.65,
    ax=axes[1, 1],
    legend=True,
)
axes[1, 1].axhline(0, color='black', lw=0.8, ls='--')
axes[1, 1].axvline(0, color='black', lw=0.8, ls='--')
axes[1, 1].set_title('Primary and secondary GR minima')
axes[1, 1].set_xlabel('best_delta_ft')
axes[1, 1].set_ylabel('second_delta_ft')
axes[1, 1].legend(loc='best', fontsize=9, title=None)

for ax in axes.ravel():
    ax.grid(True, alpha=0.25)

fig.suptitle('Prefix-only GR shift landscape', fontsize=15)
fig.savefig(OUT_DIR / 'fig_prefix_gr_shift_landscape.png', dpi=160, bbox_inches='tight')
plt.show()


> **Caution.** Do not copy the shifted GR minimum as a target.  Use the landscape as a confidence diagnostic.  Most wells are centered near $\delta=0$; the valuable cases are the minority where competing minima move away from zero, because those are where hard reforking on a single GR minimum is fragile.


## 9.1 Heel calibration: the GR-argmin fallacy is mostly a calibration artifact

The prefix shift scan above is intentionally conservative: it asks whether raw GR matching can be trusted as a hard depth label.  The stronger result from the public discussion is that the datum becomes much more recoverable after a legal heel calibration.

On known-heel rows, fit an affine gain/offset between the horizontal GR and the typewell GR sampled at known `TVT_input`:

$$
(\alpha,\beta)=
\arg\min_{\alpha,\beta}
\sum_{i\in\mathcal{P}_w}
\left(GR^h_{w,i}-\left[\alpha\,GR^{tw}_w(T^{input}_{w,i})+\beta\right]\right)^2.
$$

Then put the horizontal GR onto the typewell scale:

$$
GR^{cal}_{w,i}=\frac{GR^h_{w,i}-\beta}{\alpha},
$$

and scan the datum shift with

$$
J_w(dz)=\frac{1}{|\mathcal{P}_w|}
\sum_{i\in\mathcal{P}_w}
\left(GR^{cal}_{w,i}-GR^{tw}_w(T^{input}_{w,i}+dz)\right)^2.
$$

Georgy's public diagnostic reports the key jump: a flat/global calibration localized only about **8%** of wells within 2 ft, while heel-calibrated GR localized about **80%** within 2 ft, close to the oracle diagnostic at about **82%**; GR-rotation denoising nudged this to about **84%**.

> **Interpretation.** The datum is not hopeless.  Heel calibration shows that many train-side wells contain enough legal prefix information to recover a plausible datum.  This should not be confused with a blanket private-safe same-well contact rule: the remaining failure mode is wells where the calibrated landscape is still multimodal or where the wrong bundle minimum wins.

This has two caveats:

| Caveat | Consequence |
|---|---|
| short, flat, or gapped heel GR | the affine fit can be degenerate; fall back to weak calibration and flag the well |
| residual bundle/tie wells | do not hard-commit; pass the split posterior to the hedge in Section 14 |

Heel calibration and hedging are therefore complementary.  Calibration recovers the resolvable datum; the hedge is the floor for the residual ambiguous wells.


## 10. What the EDA establishes

> **Bridge.** The next sections are not a separate story.  They are the error model implied by the EDA above.

| EDA finding | Consequence |
|---|---|
| The prediction zone is one long tail block per well. | Evaluate and reason at the well level. |
| `TVT_input` matches train `TVT` on the known prefix. | The prefix anchor is a clean legal coordinate. |
| The constant anchor is strong, while naive prefix slope can fail badly. | Hold flat wells conservatively and treat slope transfer as risky. |
| GR/typewell alignment is real but noisy. | Use GR as a likelihood or confidence signal, not as a hard oracle. |
| Spatial and formation-surface structure is coherent. | Use spatial geology as a smooth prior, not as a target lookup. |

The remaining question is how much error belongs to datum, ambiguous mode choice, and residual per-well shape.


## 10.1 EDA-to-feature map

The old EDA notebook was large because it followed a full chain:

$$
\text{EDA observation}\rightarrow\text{geologic interpretation}\rightarrow\text{target-free estimator}\rightarrow\text{validation or ablation}.
$$

The working-note version keeps the same chain, but avoids embedding a full submission engine.  The important addition is the last column of thought: each observation must either become a direct estimator, a gated feature, a public-only override, or a do-not-use warning.

| EDA observation | Geological interpretation | Target-free estimator or diagnostic |
|---|---|---|
| last-known anchor is strong | hidden tail often starts near the prefix datum | residual target, fade/hold logic |
| GR has gaps and noisy motifs | observation likelihood should be well-specific | interpolated GR, prefix residual scale, PF likelihood |
| typewell alignment can be multimodal | hard minimum can be wrong | shift landscape, posterior hedge |
| `TVT + Z` is surface-like | formation surfaces carry datum/trend prior | spatial surface proxy, formation formula |
| estimators disagree on hard wells | disagreement is uncertainty | gated blend, posterior spread, pairwise gaps |
| long tails dominate score | slope error accumulates | datum-fixed shape ladder, row/well RMSE split |


## 10.2 What this note adds beyond the original EDA

The original EDA notebook was broader: it included data inspection, feature engineering, validation checks, model blocks, and submission engines.  This working note is meant to be its analytical upper layer: keep the useful EDA, remove leaderboard plumbing, and add what we learned later.

| Original EDA block | Keep / revise / remove | Updated interpretation |
|---|---|---|
| file inventory and schema checks | keep | the well-level data contract is the foundation of every later claim |
| leakage tables | keep and sharpen | separate direct target leakage, fold leakage, and public-overlap dependence |
| `last_known_TVT` residual target | keep | it is still the right coordinate system for most models and diagnostics |
| GR/typewell alignment | revise | use as a likelihood landscape, not as a hard argmin depth selector |
| formation surfaces | keep with caution | use as smooth spatial priors or fold-safe estimates, never direct hidden-test labels |
| PF/beam/selectors | revise | treat them as target-free path estimators with uncertainty, not as deterministic truth engines |
| same-well physical/contact shortcuts | isolate | useful public-aggressive policy, but not evidence of unseen-well robustness |
| model stacks and final submission engines | remove from this note | leaderboard recipes obscure the more durable EDA logic |
| score refork comparisons | revise | public-score micro-deltas are noisy unless the method change exceeds the refork band |

> **Updated thesis.** The competition is best understood as target-free path recovery under three uncertainties: datum, mode, and shape.  The old EDA established the signals; the later discussion taught us how not to over-read them.


In [ ]:
show_public_figure('ROGII_Graph_Fig9.png')

**Figure. EDA-driven feature engineering pipeline.**  The feature logic follows directly from geosteering observations: anchor persistence, GR correlation, formation-surface geometry, state-space continuity, selector regimes, and estimator disagreement.

In [ ]:
show_public_figure('ROGII_Graph_Fig10.png')

**Figure. Physical-estimator disagreement as uncertainty.**  PF, beam, formation, and self-correlation paths can diverge on hard wells.  The spread itself is an uncertainty signal, not just a nuisance.

**Figure D. Evidence-to-architecture ladder.**  
The working note converts data-contract observations into modeling decisions. Each diagnostic produces a target-free estimator, a policy label, or a validation guardrail, and these components finally define public-aggressive, hybrid, and private-safe submission profiles.

In [ ]:
show_public_figure('ROGII_WorkingNote_FIG_D.png')

<!-- FIG_D_READING_GUIDE -->
**How to read Figure D.**  The ladder is the audit trail from observation to action.  Each diagnostic must pass through three questions before it becomes a submission component:

$$
\text{observation}\rightarrow\text{estimator}\rightarrow\text{validation}\rightarrow\text{profile policy}.
$$

For example, a multimodal GR/typewell landscape is not a label.  It becomes a likelihood surface $\ell_w(\delta)$, then a posterior-like weight

$$
q_w(\delta)\propto \exp\{-\ell_w(\delta)/\tau\},
$$

and only then a guarded estimator or hedge.  The value of the ladder is that it prevents a good-looking public signal from being silently promoted into a private-safe assumption.

## 11. Error anatomy as an EDA summary

> **Takeaway.** Once the object is well-level, the error decomposes into recoverable datum, residual tied datum, and shape.

Write a tail prediction for well $w$ as

$$
\hat T_w(s) = \hat D_w + \hat\phi_w(s), \qquad s \in [0,1],
$$

where $\hat D_w$ is the per-well datum and $\hat\phi_w(s)$ is the tail-relative shape.

For bimodal wells, suppose two datums $a$ and $b$ are plausible:

$$
T \in \{a,b\}, \qquad \Pr(T=a)=p.
$$

Under squared loss, the Bayes estimator is the posterior mean:

$$
\hat T = p a + (1-p)b,
$$

with conditional variance

$$
\operatorname{Var}(T \mid a,b,p) = p(1-p)(a-b)^2.
$$

So detection and hedging are the same task: estimate $p$, then emit the posterior mean.  If the evidence has no mode-discriminating signal, $p \approx 1/2$ and the midpoint is optimal.


The updated reading is asymmetric.  Heel calibration makes the datum legally recoverable for a large majority of wells, but the missed datum wells are high-leverage because a bundle-scale offset affects the whole tail.  That is why a count statement such as “about 80% localized” can coexist with an MSE statement such as “datum residual dominates recoverable MSE.”


**Figure C. Error anatomy of hidden-tail TVT prediction.**  
The error is not a single scalar offset problem. After anchoring the last known TVT, the remaining error can come from datum localization, wrong stratigraphic mode selection, and hidden-tail shape recovery. The modeling architecture should address these components separately.

In [ ]:
show_public_figure('ROGII_WorkingNote_FIG_C.png')

<!-- FIG_C_READING_GUIDE -->
**How to read Figure C.**  The prediction error can be read as three coupled terms:

$$
T_w(s)-\hat T_w(s)
= \underbrace{(D_w-\hat D_w)}_{\text{datum}}
+ \underbrace{(M_w-\hat M_w)}_{\text{mode}}
+ \underbrace{(\phi_w(s)-\hat\phi_w(s))}_{\text{shape}}.
$$

The notation is schematic: $M_w$ represents the stratigraphic mode or contact bundle rather than a numeric scalar.  The modeling implication is concrete, though.  Anchor/contact/formation logic should handle datum, GR likelihood and PF/beam search should handle mode uncertainty, and projection or residual models should handle the remaining tail shape.

## 12. What the community evidence already closes

The strongest public discussion has narrowed the problem to a few stable claims.

| Claim | Evidence | Consequence |
|---|---|---|
| Datum recovery is legally strong but not perfect. | Georgy's heel-calibrated scan reports roughly 8% to 80% within 2 ft, near the oracle diagnostic. | Recover the resolvable datum first; do not treat all datum error as irreducible. |
| Tie-breaking is weak on bundle-gap wells. | souldrive reported secondary closer than primary about $51.2\%$ vs $48.8\%$, with cost-margin correlation $r=+0.054$ and $p=0.30$. | Treat GR minima as a likelihood landscape, not a mode label. |
| Refork variance is real. | Byte-identical or near-identical notebooks moved by several hundredths on the public board. | Small score deltas inside that band are not strategy evidence. |

> **Interpretation.** The signal is strong enough to recover most datums, but often not strong enough to justify a hard commitment between two plausible residual modes.


## 12.1 The don't-refork doctrine

> **Takeaway.** A better public score is not automatically a better idea.

A public notebook score can move because the method improved, because stochastic PF/beam choices changed, because the public split favored a variant, or because an overlap-specific branch happened to line up.  A useful decomposition is

$$
\Delta_{LB}
=
\Delta_{method}
+
\epsilon_{seed}
+
\epsilon_{split}
+
\epsilon_{overlap}
+
\epsilon_{implementation}.
$$

A refork is evidence only when the method signal is larger than the noise band:

$$
|\Delta_{method}| \gg
\operatorname{sd}(\epsilon_{seed}+\epsilon_{split}+\epsilon_{overlap}).
$$

If the score change is inside the observed refork band, the right response is not another blind fork.  The right response is to ask which structural component changed:

| Apparent gain | Useful question |
|---|---|
| PF seed or beam setting changed | Did the prediction path move on known hard wells, or only the public score? |
| overlap branch changed | Is this public-aggressive or private-robust? |
| stronger GR match chosen | Does the prefix shift diagnostic say that lower GR cost means correct depth? |
| model package mixed in | Is it a small correction on uncertain wells, or a global drift that breaks flat wells? |
| postprocess changed | Did it improve long drifting wells without damaging flat anchor wells? |

This is the practical meaning of “don't refork”: do not chase leaderboard micro-noise when the diagnostic does not identify a real error source.


A concrete public-board example is large enough to matter: byte-identical notebooks scored about **7.201-7.286**, and a wider set identical up to comment encoding reached about **7.168**.  A public delta of only a few hundredths can therefore be PF reseeding rather than method skill.


## 12.2 GR matching is a likelihood signal, not a hard label

The warning is not that GR matching is useless.  The warning is narrower and more important: GR argmin is usually informative, but on high-impact multimodal wells it can choose the wrong datum.

The dangerous assumption is

$$
\arg\min_{\delta} C_w(\delta)
\approx
\arg\min_{\delta} |(T^{input}+\delta)-T^{true}|.
$$

The prefix shift scan is a direct counterexample test.  On prefix rows, $T^{input}$ is known.  If the GR cost minimum moves away from $\delta=0$ there, then lower GR cost is not a reliable depth label for that well.

The safer interpretation is posterior-like:

$$
q_w(\delta) \propto \exp\left(-\frac{C_w(\delta)^2}{2\sigma_w^2}\right),
$$

where $\sigma_w$ is related to prefix GR residual scale.  A narrow, centered posterior supports a hard-ish estimate.  A broad or bimodal posterior supports hedging.

That single change of interpretation turns a brittle selector into a risk-aware estimator.

At the row level, GR should be read as a likelihood signal rather than a label assignment:

$$
p(T_i=t \mid G_i) \propto
\exp\left[-\frac{1}{2}\left(\frac{G_i-G^{type}_w(t)}{\sigma_{G,w}}\right)^2\right].
$$

This is different from the hard rule:

$$
T_i = \arg\min_t |G_i-G^{type}_w(t)|.
$$

**Modeling action.**  Use PF likelihood weighting, beam ensembles, and posterior averaging.  Avoid turning a single GR argmin depth into a direct target label.



## 13. Where the recoverable error mass sits: datum residual vs shape

> **Takeaway.** Oracle rungs are ceilings, not submission features.  In MSE, the residual datum term dominates the recoverable error mass; per-well shape remains real but secondary.

Hold a datum estimate fixed, then ask how much error remains under increasingly permissive shape assumptions.  This is a train-side measurement ladder, not a claim that every rung is available at inference time:

$$
\operatorname{RMSE}_{\text{datum-fixed}} =
\left[
\frac{1}{N}\sum_w\sum_{i \in \mathcal{E}_w}
\left(y_{w,i} - D_w^{\text{anchor}} - \hat\phi_w(s_i)\right)^2
\right]^{1/2}.
$$

| Rung | Shape assumption | Role |
|---|---|---|
| Datum only | $\hat\phi(s)=0$ | Measures flat-tail error under the anchor datum. |
| Per-well constant oracle | intercept only | Measures the best possible constant datum for each tail. |
| Prefix slope | visible-heel slope extrapolation | Negative diagnostic; legal but intentionally fragile. |
| Oracle slope | $\hat\phi(s)=\beta s$ | Measures value of perfect tail slope. |
| Oracle quadratic | $\hat\phi(s)=\beta_1s+\beta_2s^2$ | Measures value of simple curvature. |
| Smooth oracle | smooth per-well shape | Upper bound on recoverable shape. |

The oracle rungs split error, but realizability depends on legal information.  The datum tie is not meaningfully realizable from the cost margin alone ($r=+0.054$), while shape is partly realizable: public tree models around 7.04 show that some offset/slope structure is learnable from legal covariates, though the oracle gap remains only a ceiling.

**Projection coordinate.**  A shape-denoising projection is most interpretable in the formation-relative coordinate

$$
U_{w,i}=T_{w,i}+Z_{w,i}-A_w,
\qquad
A_w=T^{input}_{w,L(w)}+Z_{w,L(w)}.
$$

A low-order path model fits $U_w(s)$ rather than raw $T_w(s)$, then converts back by

$$
\hat T_{w,i}=A_w+\hat U_w(s_i)-Z_{w,i}.
$$

This keeps the projection tied to the anchor and trajectory instead of becoming a free target-space smoother.

**Diagnostic conclusion.**  Shape matters, but the MSE split shows that residual datum misses are the larger recoverable error mass.

**Modeling action.**  Try low-order slope/curvature projection and PF path smoothing as guarded postprocesses.

**Policy.**  Private-safe only if fitted from prefix, trajectory, and current prediction path; not from hidden labels.

**Guardrail.**  Evaluate well-level hurt/gain and disable projection when it over-smooths sharp wells.


## 13.1 Contact-path prefix holdout check

The contact-path datum diagnostic is made stricter than a fitted-prefix score.  A contact path can always look good if the offset is fit and evaluated on the same prefix rows.  The check used here is therefore:

1. fit the contact offset on the early prefix;
2. evaluate the resulting path on the later prefix;
3. call a well `heel_localized` only when this prefix-holdout RMSE is small.

In notation, if $\mathcal{P}^{fit}_w$ is the early prefix and $\mathcal{P}^{val}_w$ is the later prefix,

$$
\hat b_w = \operatorname{mean}_{i\in\mathcal{P}^{fit}_w}\left(T^{input}_{w,i}-T^{contact}_{w,i}\right),
$$

and

$$
RMSE^{holdout}_w =
\left[
\frac{1}{|\mathcal{P}^{val}_w|}
\sum_{i\in\mathcal{P}^{val}_w}
\left(T^{input}_{w,i}-T^{contact}_{w,i}-\hat b_w\right)^2
\right]^{1/2}.
$$

This does not make contact logic private-safe.  It only prevents the diagnostic from overstating localization quality by fitting and scoring on the same rows.

It verifies prefix stability, not hidden-tail correctness.  On train wells, a stronger diagnostic can additionally check the contact-path datum against the hidden tail labels; on test wells that tail check is unavailable, so the prefix-holdout result should be read as a localization sanity check rather than proof of private-safe tail recovery.

Because the reference contact is selected from candidate contacts using prefix evidence, this diagnostic is slightly optimistic.  It is still useful as a train-side consistency check, but it should not be read as an unseen-well validation score.

The `773/773` prefix-localized result and the roughly `80%` heel-calibration localization result are also different objects.  The former is same-well contact-path prefix consistency under this train-side diagnostic; the latter comes from the later GR-calibration/bundle discussion.  They should not be merged into one claim.


In [ ]:
DEFAULT_REF_COLS = ('EGFDU', 'EGFDL', 'ANCC', 'ASTNU', 'ASTNL', 'BUDA')


def rmse_sse(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    m = np.isfinite(y) & np.isfinite(p)
    if int(m.sum()) == 0:
        return np.nan, 0, 0.0
    e = y[m] - p[m]
    return float(np.sqrt(np.mean(e * e))), int(m.sum()), float(np.sum(e * e))


def robust_poly_predict(x_known, y_known, x_all, deg=6):
    x_known = np.asarray(x_known, dtype=float)
    y_known = np.asarray(y_known, dtype=float)
    x_all = np.asarray(x_all, dtype=float)
    m = np.isfinite(x_known) & np.isfinite(y_known)
    xk = x_known[m]
    yk = y_known[m]
    if len(xk) < 3:
        fill = float(np.nanmedian(yk)) if len(yk) else np.nan
        return np.full_like(x_all, fill, dtype=float)
    d = min(max(1, int(deg)), len(xk) - 1)
    x0 = float(np.nanmin(xk))
    xs = float(np.nanmax(xk) - x0)
    if not np.isfinite(xs) or xs < 1e-9:
        xs = 1.0
    xx = (xk - x0) / xs
    xa = (x_all - x0) / xs
    coef = np.polyfit(xx, yk, d)
    for _ in range(4):
        fit = np.polyval(coef, xx)
        r = yk - fit
        sc = float(np.nanmedian(np.abs(r - np.nanmedian(r)))) * 1.4826 + 1e-6
        w = 1.0 / (1.0 + (r / (2.5 * sc)) ** 2)
        coef = np.polyfit(xx, yk, d, w=w)
    return np.polyval(coef, xa).astype(float)


def contact_path_from_prefix(hw, tw, ref_col, split_frac=0.70):
    if ref_col not in hw.columns or not {'Z', 'TVT_input'}.issubset(hw.columns):
        return None
    if not {'Geology', 'TVT'}.issubset(tw.columns):
        return None
    tw_g = tw.dropna(subset=['Geology', 'TVT']).copy()
    vals = pd.to_numeric(tw_g.loc[tw_g['Geology'].astype(str) == str(ref_col), 'TVT'], errors='coerce').dropna()
    if vals.empty:
        return None
    ref_tvt = float(vals.min())
    z = pd.to_numeric(hw['Z'], errors='coerce').to_numpy(dtype=float)
    form = pd.to_numeric(hw[ref_col], errors='coerce').to_numpy(dtype=float)
    raw = ref_tvt - (z - form)
    known = pd.to_numeric(hw['TVT_input'], errors='coerce').to_numpy(dtype=float)
    known_idx = np.flatnonzero(np.isfinite(raw) & np.isfinite(known))
    if int(len(known_idx)) < 20:
        return None

    # Fit on the early prefix and validate on the later prefix when enough rows exist.
    if len(known_idx) >= 50:
        cut = int(np.clip(np.floor(len(known_idx) * float(split_frac)), 20, len(known_idx) - 10))
        fit_idx = known_idx[:cut]
        val_idx = known_idx[cut:]
    else:
        fit_idx = known_idx
        val_idx = np.array([], dtype=int)

    if int(len(fit_idx)) < 20:
        return None
    offset = float(np.nanmean(known[fit_idx] - raw[fit_idx]))
    pred = raw + offset

    fitted_rmse, fitted_rows, _ = rmse_sse(known[known_idx], pred[known_idx])
    if len(val_idx):
        holdout_rmse, holdout_rows, _ = rmse_sse(known[val_idx], pred[val_idx])
    else:
        holdout_rmse, holdout_rows = np.nan, 0
    score_rmse = holdout_rmse if np.isfinite(holdout_rmse) else fitted_rmse
    return {
        'ref_col': ref_col,
        'pred': pred.astype(float),
        'prefix_rmse': float(fitted_rmse),
        'prefix_rows': int(fitted_rows),
        'prefix_holdout_rmse': float(holdout_rmse) if np.isfinite(holdout_rmse) else np.nan,
        'prefix_holdout_rows': int(holdout_rows),
        'score_rmse': float(score_rmse),
    }


def best_contact_path(hw, tw, ref_cols=DEFAULT_REF_COLS):
    best = None
    for ref in ref_cols:
        item = contact_path_from_prefix(hw, tw, ref)
        if item is None or not np.isfinite(item['score_rmse']):
            continue
        if best is None or item['score_rmse'] < best['score_rmse']:
            best = item
    return best

def prefix_slope(hw, tail_rows=80, slope_clip=0.05):
    idx = np.flatnonzero(hw['TVT_input'].notna().to_numpy())
    if len(idx) < 5:
        return np.nan, np.nan
    idx = idx[-min(int(tail_rows), len(idx)):]
    md = pd.to_numeric(hw.loc[idx, 'MD'], errors='coerce').to_numpy(dtype=float)
    tvt = pd.to_numeric(hw.loc[idx, 'TVT_input'], errors='coerce').to_numpy(dtype=float)
    m = np.isfinite(md) & np.isfinite(tvt)
    md, tvt = md[m], tvt[m]
    if len(md) < 5:
        return np.nan, np.nan
    order = np.argsort(md)
    md, tvt = md[order], tvt[order]
    dmd = np.diff(md)
    dtvt = np.diff(tvt)
    ok = np.isfinite(dmd) & np.isfinite(dtvt) & (np.abs(dmd) > 1e-9)
    rates = dtvt[ok] / dmd[ok]
    rates = rates[np.isfinite(rates)]
    if len(rates) < 3:
        return np.nan, np.nan
    lo, hi = np.nanpercentile(rates, [10, 90])
    core = rates[(rates >= lo) & (rates <= hi)]
    raw = float(np.nanmedian(core if len(core) else rates))
    used = float(np.clip(raw, -slope_clip, slope_clip)) if slope_clip and slope_clip > 0 else raw
    return raw, used


def normalized_tail_coordinate(md):
    md = np.asarray(md, dtype=float)
    s = md - np.nanmin(md)
    den = float(np.nanmax(s))
    return s / den if np.isfinite(den) and den > 0 else np.linspace(0.0, 1.0, len(md))


In [ ]:
def decompose_well(wid, train_dir, prefix_rmse_limit=1.0, min_eval_rows=10, min_known_rows=30, tail_rows=80, smooth_deg=6, slope_clip=0.05):
    hw_path = train_dir / f'{wid}__horizontal_well.csv'
    tw_path = train_dir / f'{wid}__typewell.csv'
    if not hw_path.exists() or not tw_path.exists():
        return None
    hw = pd.read_csv(hw_path)
    tw = pd.read_csv(tw_path)
    if not {'TVT', 'TVT_input', 'MD'}.issubset(hw.columns):
        return None
    known_mask = hw['TVT_input'].notna().to_numpy()
    eval_mask = hw['TVT_input'].isna().to_numpy()
    if int(known_mask.sum()) < min_known_rows or int(eval_mask.sum()) < min_eval_rows:
        return None
    contact = best_contact_path(hw, tw)
    if contact is None:
        return None
    eval_idx = np.flatnonzero(eval_mask)
    md_eval = pd.to_numeric(hw.loc[eval_idx, 'MD'], errors='coerce').to_numpy(dtype=float)
    y = pd.to_numeric(hw.loc[eval_idx, 'TVT'], errors='coerce').to_numpy(dtype=float)
    contact_eval = np.asarray(contact['pred'], dtype=float)[eval_idx]
    if np.isfinite(contact_eval[0]):
        datum = float(contact_eval[0])
        datum_source = 'contact_first_eval'
    else:
        last_known = pd.to_numeric(hw.loc[known_mask, 'TVT_input'], errors='coerce').dropna()
        datum = float(last_known.iloc[-1]) if len(last_known) else float(np.nanmedian(y))
        datum_source = 'last_known_fallback'
    datum_pred = np.full(len(y), datum, dtype=float)
    raw_slope, used_slope = prefix_slope(hw, tail_rows=tail_rows, slope_clip=slope_clip)
    if np.isfinite(used_slope):
        line_pred = datum + used_slope * (md_eval - md_eval[0])
    else:
        line_pred = datum_pred.copy()
    s = normalized_tail_coordinate(md_eval)
    residual = y - datum

    def _basis_oracle(columns):
        basis = np.vstack(columns).T.astype(float)
        m = np.isfinite(residual) & np.all(np.isfinite(basis), axis=1)
        if int(m.sum()) < basis.shape[1] + 2:
            return datum_pred.copy()
        coef, *_ = np.linalg.lstsq(basis[m], residual[m], rcond=None)
        return datum + basis @ coef

    oracle_slope = _basis_oracle([s])
    oracle_quadratic = _basis_oracle([s, s ** 2])
    smooth_residual = robust_poly_predict(s, residual, s, deg=smooth_deg)
    smooth_pred = datum + smooth_residual
    oracle_constant = np.full(len(y), float(np.nanmean(y)), dtype=float)
    preds = {
        'datum_only': datum_pred,
        'oracle_constant': oracle_constant,
        'prefix_slope': line_pred,
        'oracle_slope_through_datum': oracle_slope,
        'oracle_quadratic_through_datum': oracle_quadratic,
        'smooth_shape_oracle': smooth_pred,
    }
    row = {
        'well': wid,
        'eval_rows': int(len(y)),
        'known_rows': int(known_mask.sum()),
        'heel_localized': bool(np.isfinite(contact.get('prefix_holdout_rmse', np.nan)) and contact['prefix_holdout_rmse'] <= prefix_rmse_limit),
        'contact_ref': contact['ref_col'],
        'contact_prefix_rmse': float(contact['prefix_rmse']),
        'contact_prefix_rows': int(contact['prefix_rows']),
        'contact_prefix_holdout_rmse': float(contact.get('prefix_holdout_rmse', np.nan)) if np.isfinite(contact.get('prefix_holdout_rmse', np.nan)) else np.nan,
        'contact_prefix_holdout_rows': int(contact.get('prefix_holdout_rows', 0)),
        'datum': float(datum),
        'datum_source': datum_source,
        'prefix_slope_raw': float(raw_slope) if np.isfinite(raw_slope) else np.nan,
        'prefix_slope_used': float(used_slope) if np.isfinite(used_slope) else np.nan,
        'prefix_slope_clip': float(slope_clip),
    }
    for name, pred in preds.items():
        r, n, sse = rmse_sse(y, pred)
        row[f'{name}_rmse'] = r
        row[f'{name}_rows'] = n
        row[f'{name}_sse'] = sse
    row['gap_datum_to_smooth_rmse'] = row['datum_only_rmse'] - row['smooth_shape_oracle_rmse']
    row['gap_slope_to_smooth_rmse'] = row['prefix_slope_rmse'] - row['smooth_shape_oracle_rmse']
    row['gap_oracle_slope_to_smooth_rmse'] = row['oracle_slope_through_datum_rmse'] - row['smooth_shape_oracle_rmse']
    row['gap_oracle_quadratic_to_smooth_rmse'] = row['oracle_quadratic_through_datum_rmse'] - row['smooth_shape_oracle_rmse']
    return row


def summarize_decomposition(rows_df, localized=None):
    df = rows_df.copy()
    subset = 'all_eligible'
    if localized is True:
        df = df[df['heel_localized']].copy()
        subset = 'heel_localized'
    elif localized is False:
        df = df[~df['heel_localized']].copy()
        subset = 'not_heel_localized'
    out = []
    for name in ['datum_only', 'oracle_constant', 'prefix_slope', 'oracle_slope_through_datum', 'oracle_quadratic_through_datum', 'smooth_shape_oracle']:
        sse = float(pd.to_numeric(df.get(f'{name}_sse'), errors='coerce').fillna(0).sum()) if len(df) else 0.0
        n = int(pd.to_numeric(df.get(f'{name}_rows'), errors='coerce').fillna(0).sum()) if len(df) else 0
        out.append({'subset': subset, 'model': name, 'pooled_rmse': math.sqrt(sse / max(n, 1)) if n else np.nan, 'rows': n, 'wells': int(len(df))})
    return pd.DataFrame(out)


In [ ]:
train_dir = DATA_ROOT / 'train'
wells = sorted(p.name.replace('__horizontal_well.csv', '') for p in train_dir.glob('*__horizontal_well.csv'))
rows = []
for i, wid in enumerate(wells, 1):
    row = decompose_well(wid, train_dir)
    if row is not None:
        rows.append(row)
shape_df = pd.DataFrame(rows)
shape_summary = pd.concat([
    summarize_decomposition(shape_df, localized=None),
    summarize_decomposition(shape_df, localized=True),
], ignore_index=True)
shape_df.to_csv(OUT_DIR / 'shape_slope_decomposition_by_well.csv', index=False)
contact_localization_report = pd.DataFrame([
    {
        'metric': 'eligible_wells',
        'value': int(len(shape_df)),
    },
    {
        'metric': 'holdout_localized_wells_rmse_le_1ft',
        'value': int(shape_df['heel_localized'].sum()) if 'heel_localized' in shape_df else 0,
    },
    {
        'metric': 'median_fitted_prefix_rmse',
        'value': float(shape_df['contact_prefix_rmse'].median()) if len(shape_df) else np.nan,
    },
    {
        'metric': 'median_holdout_prefix_rmse',
        'value': float(shape_df['contact_prefix_holdout_rmse'].median()) if len(shape_df) else np.nan,
    },
])
contact_localization_report.to_csv(OUT_DIR / 'contact_prefix_holdout_report.csv', index=False)
shape_summary.to_csv(OUT_DIR / 'shape_slope_decomposition_summary.csv', index=False)

recovery_rows = []
for subset, grp in shape_summary.groupby('subset'):
    by_model = grp.set_index('model')['pooled_rmse']
    datum_rmse = float(by_model.get('datum_only', np.nan))
    smooth_rmse = float(by_model.get('smooth_shape_oracle', np.nan))
    denom = datum_rmse - smooth_rmse
    for _, row in grp.iterrows():
        recovered = 100.0 * (datum_rmse - float(row['pooled_rmse'])) / denom if np.isfinite(denom) and abs(denom) > 1e-12 else np.nan
        recovery_rows.append({
            'subset': subset,
            'model': row['model'],
            'pooled_rmse': float(row['pooled_rmse']),
            'shape_gap_recovered_pct': recovered,
            'rows': int(row['rows']),
            'wells': int(row['wells']),
        })
shape_recovery = pd.DataFrame(recovery_rows)
shape_recovery.to_csv(OUT_DIR / 'shape_gap_recovery_report.csv', index=False)


mse_rows = []
for subset, grp in shape_summary.groupby('subset'):
    by_model = grp.set_index('model')['pooled_rmse']
    datum_rmse = float(by_model.get('datum_only', np.nan))
    const_rmse = float(by_model.get('oracle_constant', by_model.get('oracle_constant_context', np.nan)))
    smooth_rmse = float(by_model.get('smooth_shape_oracle', np.nan))
    datum_mse = datum_rmse ** 2
    const_mse = const_rmse ** 2
    smooth_mse = smooth_rmse ** 2
    recoverable = datum_mse - smooth_mse
    datum_residual_mse = datum_mse - const_mse
    pure_shape_mse = const_mse - smooth_mse
    mse_rows.append({
        'subset': subset,
        'datum_only_rmse': datum_rmse,
        'oracle_constant_rmse': const_rmse,
        'smooth_shape_oracle_rmse': smooth_rmse,
        'datum_residual_mse': datum_residual_mse,
        'pure_shape_mse': pure_shape_mse,
        'recoverable_mse': recoverable,
        'datum_residual_pct': 100.0 * datum_residual_mse / recoverable if np.isfinite(recoverable) and recoverable > 0 else np.nan,
        'pure_shape_pct': 100.0 * pure_shape_mse / recoverable if np.isfinite(recoverable) and recoverable > 0 else np.nan,
    })
mse_decomposition_report = pd.DataFrame(mse_rows)
mse_decomposition_report.to_csv(OUT_DIR / 'mse_decomposition_report.csv', index=False)

display(contact_localization_report)
display(shape_summary)
display(shape_recovery.round(2))
display(mse_decomposition_report.round(2))

mse_key_summary = mse_decomposition_report[mse_decomposition_report['subset'].eq('all_eligible')].copy()
if mse_key_summary.empty:
    mse_key_summary = mse_decomposition_report.head(1).copy()

mse_key_summary_display = mse_key_summary[[
    'subset',
    'datum_only_rmse',
    'oracle_constant_rmse',
    'smooth_shape_oracle_rmse',
    'datum_residual_pct',
    'pure_shape_pct',
]].copy()
print('Key MSE split computed from this run:')
display(mse_key_summary_display.round(2))
if len(mse_key_summary):
    _r = mse_key_summary.iloc[0]
    print(
        'Computed recoverable-MSE split ({subset}): datum residual {datum:.1f}% / pure shape {shape:.1f}%.'.format(
            subset=_r['subset'],
            datum=float(_r['datum_residual_pct']),
            shape=float(_r['pure_shape_pct']),
        )
    )
    from IPython.display import Markdown
    display(Markdown(
        '**Key result.** In this recompute, recoverable MSE splits into '
        f'**{float(_r["datum_residual_pct"]):.1f}% datum residual** and '
        f'**{float(_r["pure_shape_pct"]):.1f}% pure shape**.'
    ))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
plot_df = shape_summary[shape_summary['subset'].eq('heel_localized')].copy()
if plot_df.empty:
    plot_df = shape_summary[shape_summary['subset'].eq('all_eligible')].copy()
order = ['datum_only', 'oracle_constant', 'oracle_slope_through_datum', 'oracle_quadratic_through_datum', 'smooth_shape_oracle', 'prefix_slope']
plot_df['model'] = pd.Categorical(plot_df['model'], categories=order, ordered=True)
plot_df = plot_df.sort_values('model')
colors = ['#5B7CFA', '#8A8F98', '#B07AA1', '#59A14F', '#0B9A74', '#E8743B']
ax.bar(plot_df['model'].astype(str), plot_df['pooled_rmse'], color=colors[:len(plot_df)])
for x, y in enumerate(plot_df['pooled_rmse']):
    ax.text(x, y + 0.5, f'{y:.2f}', ha='center', va='bottom', fontsize=11)
ax.set_ylabel('pooled RMSE (ft, lower is better)')
ax.set_title('Datum residual and shape decomposition')
ax.set_xlabel('shape assumption')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=22, ha='right')
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig_shape_slope_decomposition.png', dpi=180, bbox_inches='tight')
plt.show()


> **Reading guide.** Read this section in MSE, not only RMSE.

The prefix-slope rung is a warning, not a solution: a local slope estimated on the visible heel often does **not** carry cleanly into the hidden tail.  It is a negative diagnostic, not a recommended ladder rung.

The additive split is:

$$
\text{datum residual MSE}=RMSE_{datum}^2-RMSE_{constant}^2,
$$

$$
\text{pure shape MSE}=RMSE_{constant}^2-RMSE_{smooth}^2.
$$

The `mse_key_summary` table printed before the plot is computed directly from `mse_decomposition_report`: the datum residual is the larger share of recoverable MSE, while pure shape is the smaller but still important share.  This reconciles two facts that can otherwise sound contradictory: a large majority of wells may be count-localized after heel calibration, yet the remaining datum misses dominate MSE because a bundle-scale datum error is copied across a long tail.  The separate `773/773` contact-prefix result is a same-well prefix-consistency diagnostic, not the same experiment as the 80% GR-calibration localization claim.


## 14. Tie wells: why hedging beats committing

> **Takeaway.** If the instrument cannot reliably choose a mode, the posterior mean is the squared-loss answer.

souldrive's tie analysis is the empirical clincher.  On wells with a clear secondary NCC minimum in the 10-25 ft bundle gap, the secondary was closer to truth about $51.2\%$ of the time versus $48.8\%$ for the primary, and the primary-secondary cost margin correlated with correctness at only

$$
r=+0.054, \qquad p=0.30.
$$

That is effectively no mode discriminator.  The work is detecting the tie and representing uncertainty, not winning the tie with a hard argmin.

If the true datum is $a$ with probability $p$ and $b$ otherwise, then the $L^2$-optimal Bayes estimator is

$$
\hat T_{\text{mean}} = p a + (1-p)b.
$$

The excess risk from committing to $a$ is

$$
R(a) - R(\hat T_{\text{mean}}) = (1-p)^2(a-b)^2,
$$

and the excess risk from committing to $b$ is

$$
R(b) - R(\hat T_{\text{mean}}) = p^2(a-b)^2.
$$

The midpoint is the special case $p=1/2$; it beats a hard mode only when $1/4 < p < 3/4$.

A calibrated two-mode posterior can be written as a softmax over the two minima.  If $J$ is a squared GR cost, a Gaussian-likelihood temperature is approximately

$$
T = \frac{2\sigma^2}{N_{eff}},
\qquad
N_{eff}\approx N\frac{1-\rho_1}{1+\rho_1},
$$

where $N_{eff}$ corrects for GR autocorrelation.  Without that correction, $p$ is usually overconfident.

The caveat is important: because the cost margin itself has almost no mode information, $p$ is a confidence weight, not a mode discriminator.  Blanket hedging all wells can hurt the easy majority; the useful hedge is gated.  The detector and the hedge are therefore the same object: keep a non-collapsing posterior only where the landscape is genuinely split.


In [ ]:
show_public_figure('ROGII_Graph_Fig5.png')


## 15. Seed/refork variance: when a public score is not evidence

> **Takeaway.** Several hundredths of public-score movement can be run noise, not method improvement.

For small public-score deltas, read the board as

$$
\Delta_{\text{observed}} = \Delta_{\text{method}} + \epsilon_{\text{PF seed}} + \epsilon_{\text{public split}}.
$$

The measured refork band was large enough to matter: byte-identical notebooks scored about **7.201-7.286**, and a wider set identical up to comment encoding reached about **7.168**.  A fork that is $0.03$ better than its neighbor can therefore be pure PF reseeding.

If $|\Delta_{\text{observed}}|$ is inside this seed/refork band, it is weak evidence.  Select final candidates by structural diagnostics and composed-prediction CV, not by public-score deltas inside the measured noise band.


## 16. What this says to do next

| Do | Why |
|---|---|
| Stop re-measuring the global floor. | The floor is already bracketed by multiple analyses. |
| Treat GR minima as likelihoods, not labels. | Cost margin has little mode-discriminating power on tie wells. |
| Use prefix shift landscapes as confidence diagnostics. | Competing shifted minima should soften the decision. |
| Detect two-mode wells and emit posterior means. | Hard mode selection is unjustified when evidence is weak. |
| Measure per-well shape recovery directly. | Datum-fixed ladders show that slope and curvature explain much of the remaining gap. |

A useful next diagnostic is a method-invariance check:

$$
r_{\text{PF}} = \operatorname{corr}\left(\ell(a)-\ell(b),\, \mathbf{1}[a\ \text{is correct}]\right)
$$

on bundle-gap wells.  If $r_{\text{PF}} \approx 0$, PF and NCC hit the same wall.  If it is positive, a non-collapsing PF carries mode information that the NCC landscape misses.


## 16.1 Validate the composed prediction system

The EDA only matters if the validation protocol preserves the information boundary.

| Guardrail | Reason |
|---|---|
| Group by `well_id` | prevents same-well autocorrelation from leaking across folds |
| evaluate hidden-tail rows only | matches the submission contract |
| keep one global postprocess policy | avoids fold-specific oracle behavior |
| separate strict/offline policies | makes batch covariates explicit instead of accidental |
| report row and well metrics | separates public-score leverage from frequent well failures |
| treat overlap shortcuts as their own policy | avoids confusing public reproduction with unseen-well robustness |

This is the main difference between a working note and a leaderboard refork.  A refork asks which setting scores best today.  A working note asks which evidence survives when the information boundary is made explicit.

**Composed-system rule.**  The validation object is not a single component.  It is the final composition:

$$
\hat T^{final}
=
\mathcal{P}_{post}
\left(
\mathcal{G}_{override}
\left(
\mathcal{P}_{shape}
\left(T_L+g_{phys}(x)+\lambda(x)g_{model}(x)\right)
\right)
\right),
$$

where $g_{phys}$ is the target-free path estimator, $g_{model}$ is a small learned residual correction, $\mathcal{P}_{shape}$ is optional shape projection, $\mathcal{G}_{override}$ is an explicit guard/override layer, and $\mathcal{P}_{post}$ is the final postprocess.  Component-level wins are not enough if the composed output loses after projection or guards.


## 16.2 Modeling lessons accumulated after the EDA

Several later experiments changed how the original EDA should be read.

| Later lesson | Practical consequence |
|---|---|
| Flat anchor wells are easy to damage. | Any correction should be gated, shrunk, or confidence-weighted near the anchor. |
| PF/beam can find strong paths but can also collapse to the wrong mode. | Preserve posterior spread; do not use only the best path. |
| Small model-package blends sometimes helped. | Treat learned residual packages as low-weight corrections, not as replacements for physical path logic. |
| Public-overlap/contact estimates can be very strong. | Keep them as an explicit public-aggressive policy rather than silently mixing them into a “robust” model. |
| Stage-level CV can mislead. | Evaluate the final composed prediction, including projection, calibration, guards, and postprocess. |
| Row-level improvements can hide well-level failures. | Report both row-weighted RMSE and per-well diagnostics. |
| Public refork scores fluctuate. | Prefer structural diagnostics over single-run leaderboard deltas. |
| Datum residual dominates recoverable MSE. | Prioritize legal heel calibration and gated datum correction before chasing shape-only refinements. |
| Shape is partly realizable but not fully oracle-realizable. | Estimate low-order well-specific slope/curvature, then guard against over-smoothing. |

A compact final prediction view is therefore

$$
\hat T
=
\underbrace{T_L}_{anchor}
+
\underbrace{g_{phys}(x)}_{target\text{-}free\ path}
+
\underbrace{\lambda(x)\,g_{model}(x)}_{small\ learned\ correction}
+
\underbrace{h_{post}(x)}_{smoothing/guard}.
$$

The main design question is not how to make $g_{model}$ large.  It is where $\lambda(x)$ should be near zero because the physical anchor is already safer.


## 16.3 Model architecture implied by the EDA

The recommended final system is not a single model.  It is a composed prediction: physical path first, learned correction second, and guarded public-overlap override only when explicitly allowed.

| Component | Role | Policy |
|---|---|---|
| anchor residual base | safe datum fallback | private-safe |
| PF/beam likelihood path | target-free physical path | private-safe if no overlap branch |
| selector regime | well-level PF/beam/hold policy | private-safe if no same-well shortcut |
| same-well physical/contact guard | public overlap exploitation | public-aggressive |
| ridge/LGB/Cat residual stack | learned correction | depends on fold and artifact policy |
| $TVT+Z-anchor$ projection | low-order shape denoising | private-safe if fit from visible/test covariates only |
| gated model correction | uncertainty-aware blend | private-safe if OOF-tuned and small |

The design goal is to keep the anchor stable, let physical estimators move only when their confidence is high, and keep learned residual corrections small on wells where the anchor is already safe.

The final system should not be described as a single model.  It is a profile-dependent composition of physical path recovery, learned residual correction, shape denoising, and optional public-overlap override.


## 16.4 Profile recommendations

The profiles are not moral categories.  They answer different validation questions.  The problem starts when a public-aggressive result is reported as if it were private-safe evidence, or when a private-safe profile is expected to reproduce an overlap-heavy public score.

### Profile formulas

A public-aggressive candidate allows the verified contact branch to act after the ordinary blend:

$$
T^{final}_{public}
=
\operatorname{GuardedOverride}
\left(\alpha T^{stack/proj}+(1-\alpha)T^{PF/selector}\right).
$$

A hybrid candidate keeps the physical estimator active but treats it as a partial correction rather than an unconditional replacement:

$$
T^{final}_{hybrid}
=
\alpha T^{stack/proj}+(1-\alpha)T^{PF/selector-private}.
$$

A private-safe candidate uses PF/beam only through a small gated correction:

$$
T^{final}_{private}
=
T^{stack/proj}+g_i\left(T^{PF/beam}-T^{stack/proj}\right),
$$

with a bounded confidence gate such as

$$
g_i=\frac{g_{max}}{1+\left(|T^{PF/beam}-T^{stack/proj}|/s\right)^2}.
$$

The formula is schematic, not a fixed submission recipe.  Its purpose is to show that the profiles differ by information policy and guard placement, not merely by a leaderboard weight.


In [ ]:
submission_profiles = pd.DataFrame([
    {
        'profile': 'public_aggressive',
        'question_answered': 'How far can public-overlap recovery go?',
        'components': 'PF/beam selector + self-verified same-well contact + guarded override',
        'same_well_physical': 'enabled with current-prefix verification',
        'validation_interpretation': 'public probe / upper-bound stress test',
    },
    {
        'profile': 'hybrid',
        'question_answered': 'Can physical recovery and small learned correction coexist?',
        'components': 'PF/beam path + residual stack + projection + small gated correction',
        'same_well_physical': 'guarded or disabled depending on objective',
        'validation_interpretation': 'balanced public/private hedge',
    },
    {
        'profile': 'private_safe',
        'question_answered': 'What survives unseen-well evaluation?',
        'components': 'fold-safe features + target-free physical estimators + tiny gated correction',
        'same_well_physical': 'disabled',
        'validation_interpretation': 'most defensible robustness probe',
    },
])

display(submission_profiles)


## 16.5 Risk register

The table below is the failure-mode companion to the profile table.  Each strong signal is useful only if its main failure mode is explicitly guarded.

| Risk | Symptom | Guard |
|---|---|---|
| GR argmin wrong mode | shifted prefix minimum or competing bundle-scale minima | posterior hedge |
| heel calibration degeneracy | short, flat, or gapped heel GR | flag well and fall back toward weak calibration |
| slope extrapolation failure | prefix slope RMSE worse than anchor | clip, shrink, or disable slope |
| public overlap over-read | large gain only when same-well shortcut is on | separate public-aggressive policy |
| seed/refork variance | score moves without structural change | deterministic seeds and prediction diff checks |
| row-level score hides well failures | small row RMSE gain but many hurt wells | well-level diagnostics |
| shape over-smoothing | projection damages sharp wells | prefix/trajectory guard |
| model package over-correction | flat wells drift away from anchor | small gated weight and flat-well damage report |
| same-well row-index mismatch | contact shortcut looks good on copied rows but fails current prefix | verify by MD interpolation against active `TVT_input` prefix |
| hidden-unsafe artifact reuse | public test cache or submission CSV is reused on hidden rerun | precompute train features/models only; build hidden-test features during rerun |

This table is the practical version of the note: every strong signal needs a corresponding failure guard.


## 16.6 What would make this a stronger working note

The most useful future additions are diagnostic, not leaderboard-driven.

| Missing diagnostic | Why it matters |
|---|---|
| PF-vs-NCC tie correlation | tests whether two independent instruments hit the same mode-selection wall |
| posterior sharpness sweep | checks whether sharper mode probabilities improve or just overcommit |
| flat-well damage report | measures whether corrections hurt wells where the anchor is already good |
| long-tail shape report | separates row-weighted gains from shape errors on high-leverage wells |
| overlap-on/off comparison | distinguishes public reproduction from private robustness |

These are the natural successors to the original EDA.  They ask why a method works, not just whether a fork scored better once.


## Appendix A. Evidence and action ledgers

These tables support the main argument without interrupting the core EDA narrative.  They separate measured facts, diagnostics, modeling inferences, public-probe hypotheses, and the action or guardrail attached to each claim.


### A.1 Evidence ladder: observation, diagnostic, estimator

A recurring source of confusion is that not every useful calculation has the same epistemic status.  This note separates three levels.

| Level | Example | What it can prove | What it cannot prove |
|---|---|---|---|
| Visible EDA observation | prefix GR residual, hidden GR gap rate, tail length | the input contains or lacks a usable target-free signal | the hidden target itself |
| Train-side diagnostic | oracle tail mean, oracle slope, datum-fixed shape ladder | where the error lives in supervised train wells | whether that oracle is available at inference |
| Submission-safe estimator | PF/beam path, prefix-calibrated surface, posterior hedge | a reproducible target-free prediction rule | that the chosen mode is correct when evidence is tied |

This is why an oracle ladder can be valuable even when it is not a feature.  It tells us where to spend modeling effort without pretending that the oracle exists in test.


### A.2 Claim ledger

Not every statement in the notebook has the same status.  The ledger below separates measured facts, diagnostics, modeling inferences, and public-probe hypotheses.  The key point is not just what the claim says, but what modeling action and policy bucket follow from it.


In [ ]:
def _metric_from_table(df_name, key_col, key_value, value_col, fmt='{:.2f}', default='not computed'):
    df = globals().get(df_name)
    try:
        if df is None or len(df) == 0:
            return default
        hit = df[df[key_col].eq(key_value)]
        if hit.empty:
            return default
        val = float(hit.iloc[0][value_col])
        return fmt.format(val)
    except Exception:
        return default


def _mse_split_metric(default='not computed'):
    df = globals().get('mse_decomposition_report')
    try:
        if df is None or len(df) == 0:
            return default
        hit = df[df['subset'].eq('all_eligible')]
        if hit.empty:
            hit = df.head(1)
        row = hit.iloc[0]
        return 'datum {:.1f}% / shape {:.1f}%'.format(
            float(row['datum_residual_pct']),
            float(row['pure_shape_pct']),
        )
    except Exception:
        return default


def _contact_metric(default='not computed'):
    df = globals().get('contact_localization_report')
    try:
        if df is None or len(df) == 0:
            return default
        vals = dict(zip(df['metric'], df['value']))
        localized = int(float(vals.get('holdout_localized_wells_rmse_le_1ft', np.nan)))
        total = int(float(vals.get('eligible_wells', np.nan)))
        med = float(vals.get('median_holdout_prefix_rmse', np.nan))
        return f'{localized}/{total}; median holdout RMSE {med:.3f}'
    except Exception:
        return default

claim_ledger = pd.DataFrame([
    {
        'claim': 'Last-known TVT is a strong datum baseline.',
        'status': 'Measured',
        'evidence': 'constant-anchor RMSE on train hidden rows',
        'metric_value': 'RMSE ' + _metric_from_table('baseline_summary', 'model', 'constant_anchor', 'pooled_rmse'),
        'modeling_action': 'predict residuals around the anchor; keep anchor fallback',
        'policy': 'private-safe target-free',
    },
    {
        'claim': 'Prefix slope extrapolation is fragile.',
        'status': 'Measured',
        'evidence': 'prefix-slope baseline can be worse than anchor',
        'metric_value': 'RMSE ' + _metric_from_table('baseline_summary', 'model', 'clipped_prefix_slope', 'pooled_rmse'),
        'modeling_action': 'use clipped/gated slope features, not raw slope prediction',
        'policy': 'private-safe with guard',
    },
    {
        'claim': 'GR/typewell matching is informative but not a label.',
        'status': 'Diagnostic',
        'evidence': 'prefix shift landscape and multimodal/tie wells',
        'metric_value': 'souldrive cost-margin r=+0.054',
        'modeling_action': 'use likelihood/posterior paths rather than hard argmin',
        'policy': 'private-safe target-free',
    },
    {
        'claim': 'TVT + Z behaves like a formation-relative coordinate.',
        'status': 'Modeling inference',
        'evidence': 'formation residual and spatial-surface diagnostics',
        'metric_value': 'see formation residual diagnostics',
        'modeling_action': 'build formation-surface, dense ANCC, and structural residual features',
        'policy': 'fold-safe train-derived',
    },
    {
        'claim': 'Same-well contact is internally consistent on train prefix.',
        'status': 'Train-side diagnostic',
        'evidence': 'contact-path prefix holdout',
        'metric_value': _contact_metric(),
        'modeling_action': 'use only as self-verifying public-aggressive override',
        'policy': 'public-aggressive overlap',
    },
    {
        'claim': 'Recoverable MSE is dominated by residual datum misses.',
        'status': 'Oracle diagnostic',
        'evidence': 'datum / constant / smooth-shape MSE decomposition',
        'metric_value': _mse_split_metric(),
        'modeling_action': 'prioritize legal heel calibration and gated datum correction',
        'policy': 'diagnostic ceiling, not direct feature',
    },
    {
        'claim': 'Smooth shape oracle is low but not fully realizable.',
        'status': 'Oracle diagnostic',
        'evidence': 'datum-fixed smooth-shape ladder',
        'metric_value': 'RMSE ' + _metric_from_table('shape_summary', 'model', 'smooth_shape_oracle', 'pooled_rmse'),
        'modeling_action': 'try low-dimensional projection and PF path smoothing with hurt guards',
        'policy': 'private-safe only if fitted from prefix/path/current prediction',
    },
])

display(claim_ledger)


### A.3 Diagnostic-to-action map

A working note should not stop at observation.  Each diagnostic below has a specific modeling consequence and a guardrail.  The purpose is to keep the notebook readable while making the implementation logic auditable: what we use directly, what we gate, what belongs only to a public-overlap profile, and what should remain a diagnostic rather than a predictor.


In [ ]:
diagnostic_action_map = pd.DataFrame([
    {
        'diagnostic': 'constant-anchor baseline',
        'finding': 'strong but incomplete datum',
        'action_class': 'Use directly',
        'modeling_action': 'residual target; anchor fallback; conservative hold weight',
        'guard': 'none beyond prefix consistency check',
    },
    {
        'diagnostic': 'prefix slope extrapolation',
        'finding': 'can be much worse than the anchor',
        'action_class': 'Use with gate',
        'modeling_action': 'slope features only; clipped/shrunk slope; no global slope-only baseline',
        'guard': 'clip slope and validate on prefix holdout',
    },
    {
        'diagnostic': 'GR shift / argmin landscape',
        'finding': 'informative but multimodal and tied wells exist',
        'action_class': 'Use with gate',
        'modeling_action': 'PF likelihood, beam ensemble, posterior averaging',
        'guard': 'avoid a single hard GR-depth label',
    },
    {
        'diagnostic': 'TVT + Z formation residual',
        'finding': 'formation-relative coordinate is structured',
        'action_class': 'Use directly / fold-safe',
        'modeling_action': 'formation surface, dense ANCC, PF-ANCC, structural residual features',
        'guard': 'OOF surfaces and imputers must be rebuilt inside folds',
    },
    {
        'diagnostic': 'PF / beam / model disagreement',
        'finding': 'large disagreement marks uncertain wells and rows',
        'action_class': 'Use with gate',
        'modeling_action': 'estimator spread features; gated blending; shrink auxiliary corrections',
        'guard': 'reduce correction weight when disagreement is large',
    },
    {
        'diagnostic': 'same-well contact prefix RMSE',
        'finding': 'can be exact on public overlap wells',
        'action_class': 'Use only for public profile',
        'modeling_action': 'self-verifying physical override',
        'guard': 'known-prefix RMSE threshold and MD-range check on the active test file',
    },
    {
        'diagnostic': 'datum-fixed shape ladder',
        'finding': 'tail shape remains after datum recovery',
        'action_class': 'Use with gate',
        'modeling_action': 'low-order projection in TVT+Z-anchor space; PF path smoothing',
        'guard': 'compare well-level hurt/gain, not only row-level RMSE',
    },
])

display(diagnostic_action_map)


## 17. Reference and acknowledgements

This note is anchored to one critical reference:

- Georgy Mamarin, [*Stop reforking: the best GR fit is the wrong depth*](https://www.kaggle.com/code/georgymamarin/stop-reforking-the-best-gr-fit-is-the-wrong-depth).

I want to thank Georgy for making the decisive discussion explicit.  The key lesson is not merely that one public fork can score better than another.  It is that the lowest-GR-cost path can still be the wrong TVT datum, so repeated reforking without an error decomposition is a weak search strategy.

I also want to credit **souldrive** for the bundle-gap tie analysis, including the $r=+0.054$ cost-margin result and the Eagle Ford / Milankovitch geological framing.  That result is what turns the hedge from an intuition into a decision-theoretic necessity.

The present working note builds around those points.  It keeps the useful target-free EDA from the earlier workflow, but reframes the later modeling problem as legal datum recovery, ambiguous-mode hedging, and residual per-well shape recovery.
